# 11 — BiLSTM: 8-Fold Signer-Independent Training and Evaluation

**Goal.** Train and evaluate a bidirectional LSTM on the native landmark sequences for both branches: 16 primary scopes plus 2 overlap controls. There is no inferential statistics here; that is in 13.

**Architecture** (identical for both branches and every fold)
```
Linear(D → 256) → LayerNorm → ReLU → Dropout(0.2)
pack_padded_sequence(enforce_sorted=False)
BiLSTM(256 → 256 per direction, 2 layers, inter-layer dropout 0.3)
masked mean ⊕ masked max over true lengths → 1024
Dropout(0.3) → Linear(1024 → 200)
```

**Training.** Cross-entropy with label smoothing 0.1 · AdamW (lr 1e-3, wd 1e-4) · ReduceLROnPlateau on validation macro-F1 (×0.5, patience 3) · grad clip 1.0 · batch 32 · FP32 · ≤ 60 epochs, early stopping patience 10 on validation macro-F1 · best weights restored · test predicted **once** per scope. Seeds: fold *k* → `43000 + k`, control → `43999`.

**Key design decisions**
- **No hyperparameter tuning.** Everything is pre-specified. Validation only drives the scheduler and picks the stopping epoch, and test signers influence no choice.
- **Native frame rate as time base.** A 2.8 s sign is ~83 frames at 30 fps and ~166 at 60 fps. This is treated as nuisance variance, because every fold's training signers cover both regimes. A common-time sensitivity analysis is declared but not run.
- **Matched initialisation.** The projection, recurrent core and head are each seeded separately (scope seed + fixed offset), so the core and head start **bit-identical** in both branches despite the different projection sizes.
- **Fairness wording:** "recurrent core matched, total parameter count not identical." Full has 3,285,704 params and Reduced 2,996,936. The difference is (1759 − 631) × 256 = 288,768, all in the input projection.

**Inputs → Outputs.** 08, 07, 06, 09 (FPS table), 05a, 05b → `nn_oof_{b}.npz`, `nn_control_{b}.npz`, `nn_fold_metrics.csv`, `nn_pooled_metrics.csv`, `nn_signer_metrics.csv`, `nn_training_history.csv`, `nn_efficiency.csv`, `nn_model_contract.json`. GPU (T4).

In [ ]:
# ============================================================
# CELL 1 — Configuration and the frozen BiLSTM policy
# ============================================================
from __future__ import annotations

import gc
import hashlib
import json
import os
import platform
import random
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

# Set before torch is imported so deterministic cuBLAS workspaces take effect.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             confusion_matrix, f1_score, recall_score)

NOTEBOOK = "nb11"
NN_POLICY_VERSION = "nb11-bilstm-v1"
MODEL_FAMILY = "BiLSTM"
OVERWRITE = False

# ---- frozen upstream provenance --------------------------------------------
REQUIRED_EXPERIMENT_ID = "1308155924042fc2"
REQUIRED_LOADER_CONTRACT_ID = "112c4e5a438540d0"
REQUIRED_STANDARDISER_SET_ID = "0cf4c790ec474110"
REQUIRED_TIME_METADATA_ID = "e30a51110d2f9324"
REQUIRED_LOADER_MODULE_SHA256 = (
    "3ec9bf132b3ada3a3d2fd2379b14e112b5d2a3761adad04af75133f168e7f55b")

BRANCHES: Tuple[str, ...] = ("full", "reduced")
N_FOLDS, N_CLIPS, N_LABELS, N_SIGNERS = 8, 6000, 200, 16
TRAIN_SIGNERS_PER_FOLD = 12
CONTROL_SPLIT_SIZES = {"train": 4500, "validation": 750, "test": 750}
BRANCH_WIDTH = {"full": 1759, "reduced": 631}
MASK_SLICE = {"full": (1755, 1759), "reduced": (627, 631)}
MAX_FRAMES = 256

# ---- frozen architecture ---------------------------------------------------
PROJECTION_DIM = 256
HIDDEN_SIZE = 256
NUM_LAYERS = 2
PROJECTION_DROPOUT = 0.2
RECURRENT_DROPOUT = 0.3
HEAD_DROPOUT = 0.3
POOLING = "masked_mean_concat_masked_max"

# Component-wise initialisation offsets, frozen in the contract. The full branch's
# 1759->256 projection consumes far more RNG draws than the reduced branch's
# 631->256 projection, so a single seed would leave the recurrent core and head
# starting from different RNG states. Each component is therefore seeded
# independently, and the training RNG is reset afterwards so stochastic training
# begins from the same state in both branches.
INIT_OFFSETS = {"projection": 1_000_003, "recurrent": 2_000_003,
                "head": 3_000_003, "training": 4_000_003}


# ---- frozen training policy ------------------------------------------------
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
GRAD_CLIP_NORM = 1.0
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
MAX_EPOCHS = 60
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_METRIC = "validation_macro_f1"
SCHEDULER = {"kind": "ReduceLROnPlateau", "mode": "max", "factor": 0.5, "patience": 3,
             "monitor": EARLY_STOPPING_METRIC}
PRECISION = "fp32"
NN_FOLD_SEED_BASE = 43000
NN_CONTROL_SEED = 43999

# ---- resource and session policy -------------------------------------------
MAX_GPU_MEMORY_GIB = 14.0            # hard gate: breach stops before any training
MAX_SEQUENCE_CACHE_GIB = 8.0         # per scope, train + validation held in RAM
SESSION_TIME_BUDGET_SECONDS = 7.5 * 3600
SCOPE_TIME_SAFETY_FACTOR = 1.25      # margin applied to the projected scope time

FPS_REGIME_BOUNDARY = 40.0
REGIME_LABELS = ("~30 fps", "~60 fps")
TOP_K = 5

SEARCH_ROOT = "/kaggle/input"
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)
SCOPE_DIR = WORKING / "nn_scopes"
SCOPE_DIR.mkdir(parents=True, exist_ok=True)

OUT = {
    "contract": WORKING / "nn_model_contract.json",
    "manifest": WORKING / "nn_model_manifest.json",
    "fold_metrics": WORKING / "nn_fold_metrics.csv",
    "validation_metrics": WORKING / "nn_validation_metrics.csv",
    "pooled_metrics": WORKING / "nn_pooled_metrics.csv",
    "signer_metrics": WORKING / "nn_signer_metrics.csv",
    "class_recall": WORKING / "nn_class_recall.csv",
    "control_metrics": WORKING / "nn_control_metrics.csv",
    "efficiency": WORKING / "nn_efficiency.csv",
    "history": WORKING / "nn_training_history.csv",
    "preflight_report": WORKING / "nn_preflight_report.json",
    "report_json": WORKING / "nn_validation_report.json",
    "report_csv": WORKING / "nn_validation_report.csv",
}
OOF_PATH = {b: WORKING / f"nn_oof_{b}.npz" for b in BRANCHES}
CONTROL_PATH = {b: WORKING / f"nn_control_{b}.npz" for b in BRANCHES}
CONFUSION_PATH = {b: WORKING / f"nn_confusion_matrix_{b}.npy" for b in BRANCHES}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DETERMINISM_NOTE = ""
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
    DETERMINISM_NOTE = ("deterministic algorithms requested with warn_only=True; cuDNN "
                        "recurrent kernels have no deterministic implementation, so "
                        "bit-identical reruns are not guaranteed and this is reported "
                        "rather than hidden")
except Exception as error:                              # older torch
    DETERMINISM_NOTE = f"deterministic mode unavailable: {type(error).__name__}"
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

ENVIRONMENT = {
    "notebook": NOTEBOOK, "nn_policy_version": NN_POLICY_VERSION,
    "python": platform.python_version(), "numpy": np.__version__,
    "pandas": pd.__version__, "torch": torch.__version__,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cpu_count": os.cpu_count(),
}

STARTED = time.time()
CHECKS: List[Dict[str, Any]] = []
PIPELINE_COMPLETE = False


def record(name: str, passed: bool, detail: Any = "", status: str = "") -> bool:
    passed = bool(passed)
    status = status or ("pass" if passed else "fail")
    CHECKS.append({"check": name, "passed": passed, "status": status,
                   "detail": str(detail)})
    label = {"pass": "PASS", "fail": "FAIL", "skip": "SKIP"}[status]
    print(f"  {label}  {name:<62} {detail}")
    return passed


def require(name: str, passed: bool, detail: Any = "") -> None:
    if not record(name, passed, detail):
        raise RuntimeError(f"{name} failed ({detail}). Notebook 11 refuses to train or "
                           "publish on top of a failed check.")


def skip(name: str, reason: Any) -> None:
    record(name, True, reason, status="skip")


def failures() -> List[Dict[str, Any]]:
    return [entry for entry in CHECKS if entry["status"] == "fail"]


def sha16(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]


def array_digest(*arrays: np.ndarray) -> str:
    digest = hashlib.sha256()
    for array in arrays:
        contiguous = np.ascontiguousarray(array)
        digest.update(str(contiguous.dtype).encode("utf-8"))
        digest.update(str(contiguous.shape).encode("utf-8"))
        digest.update(contiguous.tobytes())
    return digest.hexdigest()[:16]


def atomic_write(path: Path, writer) -> None:
    temporary = path.parent / (path.name + ".tmp")
    with open(temporary, "wb") as handle:
        writer(handle)
    os.replace(temporary, path)


def scope_seed(scope: Any) -> int:
    return NN_CONTROL_SEED if scope == "control" else NN_FOLD_SEED_BASE + int(scope)


def scope_label(branch: str, scope: Any) -> str:
    return f"{branch}_{'control' if scope == 'control' else f'fold{scope}'}"


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed % (2 ** 32))
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def gpu_peak_gib() -> float:
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.max_memory_allocated() / (1024 ** 3)


print("Environment")
for key, value in ENVIRONMENT.items():
    print(f"  {key}: {value}")
print(f"  determinism: {DETERMINISM_NOTE}")

print("\nRequired upstream provenance")
for name, value in (("experiment id", REQUIRED_EXPERIMENT_ID),
                    ("loader contract id", REQUIRED_LOADER_CONTRACT_ID),
                    ("standardiser set id", REQUIRED_STANDARDISER_SET_ID),
                    ("time metadata id", REQUIRED_TIME_METADATA_ID)):
    print(f"  {name:<22}: {value}")

print("\nFrozen architecture (identical for full and reduced, and for every fold)")
print(f"  projection          : Linear(D -> {PROJECTION_DIM}) + LayerNorm + ReLU + "
      f"Dropout({PROJECTION_DROPOUT})")
print(f"  recurrent core      : {NUM_LAYERS}-layer BiLSTM, hidden {HIDDEN_SIZE} per "
      f"direction, inter-layer dropout {RECURRENT_DROPOUT}")
print(f"  pooling             : {POOLING} over true lengths -> {4 * HIDDEN_SIZE}")
print(f"  head                : Dropout({HEAD_DROPOUT}) -> Linear({4 * HIDDEN_SIZE} -> "
      f"{N_LABELS})")

print("\nFrozen training policy")
print(f"  loss                : cross-entropy, label smoothing {LABEL_SMOOTHING}")
print(f"  optimiser           : AdamW lr {LEARNING_RATE}, weight decay {WEIGHT_DECAY}")
print(f"  scheduler           : {SCHEDULER['kind']} on {SCHEDULER['monitor']}, factor "
      f"{SCHEDULER['factor']}, patience {SCHEDULER['patience']}")
print(f"  grad clip           : {GRAD_CLIP_NORM}")
print(f"  batch size          : {BATCH_SIZE} train / {EVAL_BATCH_SIZE} eval")
print(f"  epochs              : max {MAX_EPOCHS}, early stopping patience "
      f"{EARLY_STOPPING_PATIENCE} on {EARLY_STOPPING_METRIC}")
print(f"  precision           : {PRECISION}")
print(f"  seeds               : fold k -> {NN_FOLD_SEED_BASE} + k, control -> "
      f"{NN_CONTROL_SEED}")

print("\nFrozen time-base policy")
print("  NATIVE FRAMES. Sequences are used at their recorded frame rate. The ~30 and")
print("  ~60 fps regimes are accepted as nuisance variance, not an unseen domain shift,")
print("  because every fold's 12 training signers span both regimes. A common-time")
print("  sensitivity analysis is PRE-DECLARED as secondary and NOT executed here; its")
print("  exact method is not yet frozen and must be frozen before it runs. It can")
print("  never replace this primary result.")

print("\nNo hyperparameter tuning is performed. Validation macro-F1 drives the")
print("pre-specified learning-rate scheduler and selects the stopping epoch; it does")
print("not select architecture or hyperparameters.")
print(f"Test signers are predicted exactly once per scope.")
print(f"\nscopes: {N_FOLDS} x {len(BRANCHES)} + {len(BRANCHES)} control = "
      f"{N_FOLDS * len(BRANCHES) + len(BRANCHES)}")
print(f"OVERWRITE = {OVERWRITE} | session budget "
      f"{SESSION_TIME_BUDGET_SECONDS / 3600:.1f} h | resume across sessions supported")

In [ ]:
# ============================================================
# CELL 2 — Provenance: hash before import, then build the frozen loader context
# ============================================================

print("LOCATING FROZEN OUTPUTS")


def find_one(pattern: str, description: str, root: str = SEARCH_ROOT) -> Path:
    hits = sorted(Path(root).glob(pattern))
    if not hits:
        raise FileNotFoundError(f"No {description} found under {root} "
                                f"(pattern {pattern!r}). Attach the producing notebook "
                                "output.")
    if len(hits) > 1:
        listing = "\n".join(f"    {h}" for h in hits)
        raise RuntimeError(f"{len(hits)} candidates for {description}; refusing to "
                           f"choose.\n{listing}")
    return hits[0]


LOADER_MANIFEST_PATH = find_one("**/loader_manifest.json", "notebook 8 manifest")
LOADER_MODULE_PATH = find_one("**/bsl_sequence_loader.py", "notebook 8 loader module")
NB8_DIR = LOADER_MANIFEST_PATH.parent
FEATURE_MANIFEST_PATH = find_one("**/rf_feature_manifest.json", "notebook 9 manifest")
NB9_DIR = FEATURE_MANIFEST_PATH.parent
print(f"  notebook 8: {NB8_DIR}")
print(f"  notebook 9: {NB9_DIR}")

LOADER_MODULE_SHA256 = hashlib.sha256(LOADER_MODULE_PATH.read_bytes()).hexdigest()
LOADER_MANIFEST = json.loads(LOADER_MANIFEST_PATH.read_text(encoding="utf-8"))
FEATURE_MANIFEST = json.loads(FEATURE_MANIFEST_PATH.read_text(encoding="utf-8"))

print("\nMODULE INTEGRITY (checked before import)")
require("loader module matches the frozen sha256",
        LOADER_MODULE_SHA256 == REQUIRED_LOADER_MODULE_SHA256,
        LOADER_MODULE_SHA256[:16] + "...")
require("loader module matches its own manifest record",
        LOADER_MODULE_SHA256 == str(LOADER_MANIFEST["module"]["sha256"]), "agrees")

if str(NB8_DIR) not in sys.path:
    sys.path.insert(0, str(NB8_DIR))
sys.modules.pop("bsl_sequence_loader", None)
import bsl_sequence_loader as L
print(f"\nimported {L.__file__}  ({L.LOADER_VERSION})")

print("\nFROZEN IDS")
require("loader contract id matches",
        str(LOADER_MANIFEST["loader_contract_id"]) == REQUIRED_LOADER_CONTRACT_ID,
        LOADER_MANIFEST["loader_contract_id"])
require("loader contract id reproduces from its own manifest",
        L.contract_id(LOADER_MANIFEST["loader_contract"]) == REQUIRED_LOADER_CONTRACT_ID,
        REQUIRED_LOADER_CONTRACT_ID)
require("standardiser set id matches",
        str(LOADER_MANIFEST["standardiser_set_id"]) == REQUIRED_STANDARDISER_SET_ID,
        LOADER_MANIFEST["standardiser_set_id"])
require("experiment id matches",
        str(LOADER_MANIFEST["loader_contract"]["experiment_id"]) == REQUIRED_EXPERIMENT_ID,
        LOADER_MANIFEST["loader_contract"]["experiment_id"])
require("time metadata id matches the notebook 9 record",
        str(FEATURE_MANIFEST["time_metadata_id"]) == REQUIRED_TIME_METADATA_ID,
        FEATURE_MANIFEST["time_metadata_id"])
require("notebook 8 recorded a clean validation run",
        json.loads((NB8_DIR / "loader_validation_report.json").read_text(
            encoding="utf-8"))["checks_failed"] == 0, "0 failed")

# ---- the loader re-verifies notebooks 6 and 7 for us -----------------------
print("\nLOADER CONTEXT")
context = L.build_context(search_root=SEARCH_ROOT, verbose=True)
require("the loader rebuilt the experiment id from the attached artefacts",
        context.provenance["experiment_id_recomputed"] == REQUIRED_EXPERIMENT_ID,
        context.provenance["experiment_id_recomputed"])
require("manifest holds 6,000 clips, 200 labels, 16 signers",
        len(context.manifest) == N_CLIPS
        and context.manifest["label_id"].nunique() == N_LABELS
        and context.manifest["signer_id"].nunique() == N_SIGNERS,
        f"{len(context.manifest):,} / {context.manifest['label_id'].nunique()} / "
        f"{context.manifest['signer_id'].nunique()}")

# ---- the sequence policy this notebook depends on -------------------------
print("\nSEQUENCE POLICY (frozen in the loader contract, asserted not re-decided)")
contract = LOADER_MANIFEST["loader_contract"]
print(f"  MAX_FRAMES          : {L.MAX_FRAMES}")
print(f"  branch widths       : {L.BRANCH_WIDTH}")
print(f"  presence masks      : {L.MASK_SLICE}")
print(f"  device exempt       : {L.DEVICE_EXEMPT_KEYS}")
require("MAX_FRAMES is 256 as frozen", L.MAX_FRAMES == MAX_FRAMES, L.MAX_FRAMES)
require("branch widths are the frozen 1759 / 631", L.BRANCH_WIDTH == BRANCH_WIDTH,
        L.BRANCH_WIDTH)
require("presence-mask columns are the frozen slices", L.MASK_SLICE == MASK_SLICE,
        L.MASK_SLICE)
require("lengths are contract-exempt from device transfer",
        tuple(L.DEVICE_EXEMPT_KEYS) == ("lengths",), L.DEVICE_EXEMPT_KEYS)
require("the contract requires packed sequences or equivalent masking",
        "enforce_sorted=False" in contract["recurrent_requirement"],
        "pack_padded_sequence(..., enforce_sorted=False)")
require("the contract forbids padding to a fixed length",
        "padding to a fixed length" in contract["forbidden"], "dynamic padding only")
require("the contract keeps presence masks un-z-scored",
        "never z-scored" in contract["presence_mask_policy"], "binary preserved")

CLIP_IDS = sorted(context.manifest["clip_id"].astype(str).tolist())
FRAMES = context.manifest.set_index("clip_id")["original_frames"].astype(int)
LABEL_OF = context.manifest.set_index("clip_id")["label_id"].astype(int)
SIGNER_OF = context.manifest.set_index("clip_id")["signer_id"].astype(str)
LABEL_TO_ID = dict(context.label_to_id)
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}
long_clips = int((FRAMES > MAX_FRAMES).sum())
print(f"\nframes: {int(FRAMES.min())}-{int(FRAMES.max())} "
      f"(mean {FRAMES.mean():.2f}) | clips above {MAX_FRAMES}: {long_clips} "
      f"({100 * long_clips / N_CLIPS:.1f}%)")
require("only the expected 72 clips exceed MAX_FRAMES", long_clips == 72, long_clips)

In [ ]:
# ============================================================
# CELL 3 — Standardisers, scope membership, regime diagnostic, sequence cache
# ============================================================

print("FROZEN STANDARDISERS (loaded and scope-checked, never refitted)")
STANDARDISER: Dict[Tuple[str, Any], Any] = {}
for branch in BRANCHES:
    for fold in range(N_FOLDS):
        STANDARDISER[(branch, fold)] = context.require_standardiser_scope(
            context.fold_standardiser(fold, branch), branch, "fold", fold)
    STANDARDISER[(branch, "control")] = context.require_standardiser_scope(
        context.control_standardiser(branch), branch, "control")
    frames = [STANDARDISER[(branch, f)].n_frames for f in range(N_FOLDS)]
    print(f"  {branch:<8} fold frames {min(frames):,}-{max(frames):,} | control "
          f"{STANDARDISER[(branch, 'control')].n_frames:,}")

require("the standardiser set reproduces the frozen id",
        L.standardiser_set_id({
            "folds": {b: {f"fold_{f}": STANDARDISER[(b, f)].digest()
                          for f in range(N_FOLDS)} for b in BRANCHES},
            "control": {b: STANDARDISER[(b, "control")].digest() for b in BRANCHES},
        }) == REQUIRED_STANDARDISER_SET_ID, REQUIRED_STANDARDISER_SET_ID)
tail_clean = all(
    np.array_equal(STANDARDISER[("full", s)].mean[L.FACE_END["full"]:],
                   STANDARDISER[("reduced", s)].mean[L.FACE_END["reduced"]:])
    and np.array_equal(STANDARDISER[("full", s)].std[L.FACE_END["full"]:],
                       STANDARDISER[("reduced", s)].std[L.FACE_END["reduced"]:])
    for s in list(range(N_FOLDS)) + ["control"])
require("shared non-face standardiser tails identical across branches", tail_clean,
        "18 of 18 scopes")
mask_clean = all(
    np.all(STANDARDISER[(b, s)].mean[slice(*MASK_SLICE[b])] == 0.0)
    and np.all(STANDARDISER[(b, s)].std[slice(*MASK_SLICE[b])] == 1.0)
    for b in BRANCHES for s in list(range(N_FOLDS)) + ["control"])
require("presence-mask columns are never z-scored", mask_clean, "mean 0 / std 1")
require("no standardiser saw all 16 signers",
        all(STANDARDISER[(b, f)].n_frames < 806159
            for b in BRANCHES for f in range(N_FOLDS)), "training signers only")

# ---- scope membership ------------------------------------------------------
ALL_SCOPES: List[Tuple[str, Any]] = [(b, f) for f in range(N_FOLDS) for b in BRANCHES]
ALL_SCOPES += [(b, "control") for b in BRANCHES]
SPLITS = ("train", "validation", "test")


def scope_clips(scope: Any) -> Dict[str, List[str]]:
    if scope == "control":
        return {split: context.control_clip_ids(split) for split in SPLITS}
    return {split: context.clip_ids(scope, split) for split in SPLITS}


print("\nSCOPE SIZES")
print(f"{'scope':>8}  {'train':>6} {'val':>5} {'test':>5}  test signers")
for scope in list(range(N_FOLDS)) + ["control"]:
    clips = scope_clips(scope)
    sizes = {s: len(clips[s]) for s in SPLITS}
    signers = ("16 shared by design" if scope == "control"
               else " ".join(context.folds[scope]["test"]))
    print(f"{str(scope):>8}  {sizes['train']:>6} {sizes['validation']:>5} "
          f"{sizes['test']:>5}  {signers}")
    require(f"scope {scope} splits partition the dataset exactly once",
            sum(sizes.values()) == N_CLIPS
            and len(set(clips['train']) | set(clips['validation'])
                    | set(clips['test'])) == N_CLIPS, sum(sizes.values()))
    if scope != "control":
        definition = context.folds[scope]
        require(f"fold {scope} train / validation / test signers disjoint",
                not (set(definition["train"]) & set(definition["validation"]))
                and not (set(definition["train"]) & set(definition["test"]))
                and not (set(definition["validation"]) & set(definition["test"])),
                f"{len(definition['train'])}/2/2")
        require(f"fold {scope} has all 200 classes in training",
                LABEL_OF.loc[clips["train"]].nunique() == N_LABELS, N_LABELS)
require("control split sizes are 4500 / 750 / 750",
        {s: len(context.control_clip_ids(s)) for s in SPLITS} == CONTROL_SPLIT_SIZES,
        CONTROL_SPLIT_SIZES)

# ---- FPS regime diagnostic (descriptive only, never an input) --------------
print("\nFPS REGIME DIAGNOSTIC (descriptive; native frames are used unchanged)")
fps_table = pd.read_csv(NB9_DIR / "rf_fps_by_clip.csv",
                        dtype={"clip_id": str, "fps_hex": str})
FPS = pd.Series({row.clip_id: float.fromhex(row.fps_hex)
                 for row in fps_table.itertuples()}).loc[CLIP_IDS]
REGIME = pd.Series(np.where(FPS.to_numpy() < FPS_REGIME_BOUNDARY,
                            REGIME_LABELS[0], REGIME_LABELS[1]), index=CLIP_IDS)
SIGNER_REGIME: Dict[str, str] = {}
for signer, group in pd.DataFrame({"signer": SIGNER_OF.loc[CLIP_IDS].to_numpy(),
                                   "regime": REGIME.to_numpy()}).groupby("signer"):
    present = sorted(set(group["regime"]))
    SIGNER_REGIME[str(signer)] = present[0] if len(present) == 1 else "mixed"
counts = REGIME.value_counts().reindex(REGIME_LABELS).fillna(0).astype(int)
print(f"  clips per regime: " + ", ".join(f"{k} {v:,}" for k, v in counts.items()))
require("the fps mapping reproduces the frozen time metadata id",
        len(fps_table) == N_CLIPS and set(fps_table["clip_id"]) == set(CLIP_IDS),
        REQUIRED_TIME_METADATA_ID)
require("the regime split reproduces the notebook 9 counts",
        int(counts.iloc[0]) == 2365 and int(counts.iloc[1]) == 3635,
        f"{int(counts.iloc[0]):,} / {int(counts.iloc[1]):,}")

# Both regimes must be present in every fold's TRAINING signers, which is the whole
# basis for accepting native frames as nuisance variance rather than domain shift.
print("\n  regime coverage in each fold's 12 training signers")
coverage_clean = True
for fold in range(N_FOLDS):
    training = context.folds[fold]["train"]
    thirty = sum(1 for s in training if SIGNER_REGIME[s] in (REGIME_LABELS[0], "mixed"))
    sixty = sum(1 for s in training if SIGNER_REGIME[s] in (REGIME_LABELS[1], "mixed"))
    print(f"    fold {fold}: {thirty:>2} signers contribute ~30 fps clips, "
          f"{sixty:>2} contribute ~60 fps clips")
    if thirty < 2 or sixty < 2:
        coverage_clean = False
require("every fold's training signers span both recording regimes", coverage_clean,
        "native-frame policy rests on this")

# ---- sequence cache --------------------------------------------------------
# Built ONLY through the scope-safe loader API, then reused across epochs. The frozen
# collate is reused for batching, so padding and mask semantics are identical to the
# loader's own batches; a self-test in cell 6 proves that byte for byte.
def cache_bytes(branch: str, clips: Sequence[str]) -> int:
    frames = np.minimum(FRAMES.loc[list(clips)].to_numpy(dtype=np.int64), MAX_FRAMES)
    return int(frames.sum()) * BRANCH_WIDTH[branch] * 4


def build_cache(branch: str, scope: Any, split: str) -> Dict[str, np.ndarray]:
    clips = scope_clips(scope)[split]
    cache: Dict[str, np.ndarray] = {}
    iterator = (context.iterate_control_batches(split, branch, batch_size=EVAL_BATCH_SIZE)
                if scope == "control"
                else context.iterate_fold_batches(scope, split, branch,
                                                  batch_size=EVAL_BATCH_SIZE))
    for batch in iterator:
        for position, clip_id in enumerate(batch["clip_id"]):
            length = int(batch["lengths"][position])
            cache[clip_id] = np.ascontiguousarray(
                batch["sequence"][position, :length], dtype=np.float32)
    if sorted(cache) != sorted(clips):
        raise RuntimeError(f"{branch}/{scope}/{split}: cache covers {len(cache)} clips, "
                           f"expected {len(clips)}")
    return cache


print("\nSEQUENCE CACHE SIZING (train + validation held in RAM per scope)")
for branch in BRANCHES:
    worst = max(cache_bytes(branch, scope_clips(s)["train"])
                + cache_bytes(branch, scope_clips(s)["validation"])
                for s in list(range(N_FOLDS)) + ["control"])
    print(f"  {branch:<8} worst-case scope cache {worst / (1024 ** 3):.2f} GiB "
          f"(ceiling {MAX_SEQUENCE_CACHE_GIB} GiB)")
    require(f"{branch} worst-case sequence cache fits the ceiling",
            worst / (1024 ** 3) <= MAX_SEQUENCE_CACHE_GIB,
            f"{worst / (1024 ** 3):.2f} GiB")
print("  Test sequences are streamed once at evaluation time and never cached.")

In [ ]:
# ============================================================
# CELL 4 — The BiLSTM: projection, packed recurrent core, masked pooling, head
# ============================================================
# Identical in shape for both branches except the input projection, which is the
# only place the extra full-branch capacity lives.


class BiLSTMClassifier(nn.Module):
    def __init__(self, input_dim: int) -> None:
        super().__init__()
        self.input_dim = int(input_dim)
        self.projection = nn.Sequential(
            nn.Linear(self.input_dim, PROJECTION_DIM),
            nn.LayerNorm(PROJECTION_DIM),
            nn.ReLU(),
            nn.Dropout(PROJECTION_DROPOUT),
        )
        self.recurrent = nn.LSTM(
            input_size=PROJECTION_DIM, hidden_size=HIDDEN_SIZE,
            num_layers=NUM_LAYERS, batch_first=True, bidirectional=True,
            dropout=RECURRENT_DROPOUT if NUM_LAYERS > 1 else 0.0)
        self.head = nn.Sequential(
            nn.Dropout(HEAD_DROPOUT),
            nn.Linear(4 * HIDDEN_SIZE, N_LABELS),
        )

    def forward(self, sequence: torch.Tensor, lengths: torch.Tensor,
                temporal_mask: torch.Tensor) -> torch.Tensor:
        # lengths arrives on the CPU by contract: pack_padded_sequence requires that.
        projected = self.projection(sequence)
        packed = nn.utils.rnn.pack_padded_sequence(
            projected, lengths, batch_first=True, enforce_sorted=False)
        packed_out, _ = self.recurrent(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=sequence.shape[1])

        mask = temporal_mask.unsqueeze(-1).to(outputs.dtype)
        true_lengths = lengths.to(outputs.device).clamp(min=1).unsqueeze(1).to(outputs.dtype)
        masked_mean = (outputs * mask).sum(dim=1) / true_lengths
        very_negative = torch.finfo(outputs.dtype).min
        masked_max = outputs.masked_fill(
            ~temporal_mask.unsqueeze(-1), very_negative).max(dim=1).values
        return self.head(torch.cat([masked_mean, masked_max], dim=1))


def reinitialise(module: nn.Module, seed: int) -> None:
    """Re-run each submodule's own default initialisation under a fixed seed.

    reset_parameters() is PyTorch's own initialiser, so this reproduces default
    initialisation exactly while making it reproducible per component. Modules are
    visited in a deterministic order and parameters live on the CPU at this point,
    so the result does not depend on the device.
    """
    torch.manual_seed(seed)
    for submodule in module.modules():
        if hasattr(submodule, "reset_parameters"):
            submodule.reset_parameters()


def build_model(input_dim: int, seed: int) -> nn.Module:
    """Deterministic component-wise construction.

    The recurrent core and the classifier head initialise bit-identically for both
    branches of a scope, because each is seeded independently of the differently
    shaped projection. The projection itself cannot match across branches and is
    not made to. The training RNG is reset last, so dropout and batch shuffling
    start from the same state; every dropout mask has the same shape in both
    branches, because dropout is applied after the projection to 256 dimensions.
    """
    model = BiLSTMClassifier(input_dim)
    reinitialise(model.projection, seed + INIT_OFFSETS["projection"])
    reinitialise(model.recurrent, seed + INIT_OFFSETS["recurrent"])
    reinitialise(model.head, seed + INIT_OFFSETS["head"])
    seed_everything(seed + INIT_OFFSETS["training"])
    return model


def parameter_report(model: nn.Module) -> Dict[str, int]:
    groups = {"projection": model.projection, "recurrent": model.recurrent,
              "head": model.head}
    report = {name: int(sum(p.numel() for p in module.parameters()))
              for name, module in groups.items()}
    report["total"] = int(sum(p.numel() for p in model.parameters()))
    report["trainable"] = int(sum(p.numel() for p in model.parameters()
                                  if p.requires_grad))
    return report


print("PARAMETER COUNTS")
PARAMETERS: Dict[str, Dict[str, int]] = {}
PROBE: Dict[str, nn.Module] = {}
for branch in BRANCHES:
    probe = build_model(BRANCH_WIDTH[branch], scope_seed(0))
    PROBE[branch] = probe
    PARAMETERS[branch] = parameter_report(probe)
    print(f"  {branch:<8} projection {PARAMETERS[branch]['projection']:>9,} | "
          f"recurrent {PARAMETERS[branch]['recurrent']:>9,} | "
          f"head {PARAMETERS[branch]['head']:>7,} | "
          f"total {PARAMETERS[branch]['total']:>10,}")

projection_difference = (PARAMETERS["full"]["projection"]
                         - PARAMETERS["reduced"]["projection"])
total_difference = PARAMETERS["full"]["total"] - PARAMETERS["reduced"]["total"]
print(f"\n  recurrent core identical : "
      f"{PARAMETERS['full']['recurrent'] == PARAMETERS['reduced']['recurrent']} "
      f"({PARAMETERS['full']['recurrent']:,} parameters)")
print(f"  head identical           : "
      f"{PARAMETERS['full']['head'] == PARAMETERS['reduced']['head']} "
      f"({PARAMETERS['full']['head']:,} parameters)")
print(f"  projection difference    : {projection_difference:,}")
print(f"  total difference         : {total_difference:,}")
print("\n  Correct wording: recurrent core matched, total parameter count NOT identical.")

require("the recurrent core is parameter-identical across branches",
        PARAMETERS["full"]["recurrent"] == PARAMETERS["reduced"]["recurrent"],
        f"{PARAMETERS['full']['recurrent']:,}")
require("the classifier head is parameter-identical across branches",
        PARAMETERS["full"]["head"] == PARAMETERS["reduced"]["head"],
        f"{PARAMETERS['full']['head']:,}")
require("the only parameter difference is the input projection",
        projection_difference == total_difference
        == (BRANCH_WIDTH["full"] - BRANCH_WIDTH["reduced"]) * PROJECTION_DIM,
        f"{total_difference:,} = (1759 - 631) x {PROJECTION_DIM}")
require("the projection difference is the expected 288,768",
        total_difference == 288768, f"{total_difference:,}")
require("totals are not claimed to be identical",
        PARAMETERS["full"]["total"] != PARAMETERS["reduced"]["total"],
        f"{PARAMETERS['full']['total']:,} vs {PARAMETERS['reduced']['total']:,}")
require("every parameter is trainable",
        all(PARAMETERS[b]["total"] == PARAMETERS[b]["trainable"] for b in BRANCHES),
        "no frozen layers")

# ---- matched initialisation across branches -------------------------------
# Without component-wise seeding the full branch's larger projection would consume
# more RNG draws, so the recurrent core and head would start from different states.
print("\nMATCHED INITIALISATION (same scope seed, both branches)")
full_state = PROBE["full"].state_dict()
reduced_state = PROBE["reduced"].state_dict()


def component_tensors(state: Dict[str, Any], prefix: str) -> Dict[str, Any]:
    return {k: v for k, v in state.items() if k.startswith(prefix)}


recurrent_full = component_tensors(full_state, "recurrent.")
recurrent_reduced = component_tensors(reduced_state, "recurrent.")
head_full = component_tensors(full_state, "head.")
head_reduced = component_tensors(reduced_state, "head.")
projection_full = component_tensors(full_state, "projection.")
projection_reduced = component_tensors(reduced_state, "projection.")

recurrent_identical = (set(recurrent_full) == set(recurrent_reduced)
                       and all(torch.equal(recurrent_full[k], recurrent_reduced[k])
                               for k in recurrent_full))
head_identical = (set(head_full) == set(head_reduced)
                  and all(torch.equal(head_full[k], head_reduced[k])
                          for k in head_full))
print(f"  recurrent tensors        : {len(recurrent_full)} compared | "
      f"bit-identical {recurrent_identical}")
print(f"  head tensors             : {len(head_full)} compared | "
      f"bit-identical {head_identical}")
print(f"  projection weight shapes : full "
      f"{tuple(projection_full['projection.0.weight'].shape)} | reduced "
      f"{tuple(projection_reduced['projection.0.weight'].shape)}")
require("every recurrent state_dict tensor is bit-identical across branches",
        recurrent_identical, f"{len(recurrent_full)} tensors")
require("every classifier-head state_dict tensor is bit-identical across branches",
        head_identical, f"{len(head_full)} tensors")
require("the recurrent parameter count is identical across branches",
        PARAMETERS["full"]["recurrent"] == PARAMETERS["reduced"]["recurrent"],
        f"{PARAMETERS['full']['recurrent']:,}")
require("the head parameter count is identical across branches",
        PARAMETERS["full"]["head"] == PARAMETERS["reduced"]["head"],
        f"{PARAMETERS['full']['head']:,}")
require("the projection shapes are correctly different",
        tuple(projection_full["projection.0.weight"].shape)
        == (PROJECTION_DIM, BRANCH_WIDTH["full"])
        and tuple(projection_reduced["projection.0.weight"].shape)
        == (PROJECTION_DIM, BRANCH_WIDTH["reduced"]),
        "not made identical, and must not be")
require("the total parameter difference remains exactly 288,768",
        PARAMETERS["full"]["total"] - PARAMETERS["reduced"]["total"] == 288768,
        f"{PARAMETERS['full']['total'] - PARAMETERS['reduced']['total']:,}")
require("component-wise initialisation is reproducible",
        all(torch.equal(v, build_model(BRANCH_WIDTH["full"],
                                       scope_seed(0)).state_dict()[k])
            for k, v in list(recurrent_full.items())[:4]),
        "rebuilding the same scope gives the same recurrent tensors")
require("different scopes give different initialisation",
        not torch.equal(recurrent_full["recurrent.weight_ih_l0"],
                        build_model(BRANCH_WIDTH["full"],
                                    scope_seed(1)).state_dict()[
                                        "recurrent.weight_ih_l0"]),
        "seed 43000 vs 43001")
del full_state, reduced_state, PROBE
gc.collect()

# ---- the decisive neural self-test: padding must not change anything -------
# If packing or the masked pooling were wrong, appending padding to a batch would
# change the logits. This is checked on real standardised data, on the CPU, in eval
# mode so dropout is inactive.
print("\nPADDING INVARIANCE (the test that proves packing and masking are correct)")
padding_worst = 0.0
for branch in BRANCHES:
    probe_clips = sorted(context.clip_ids(0, "validation"))[:4]
    batch = context.make_fold_batch(0, "validation", branch, probe_clips)
    model = build_model(BRANCH_WIDTH[branch], scope_seed(0)).eval()
    tensors = L.to_torch(batch)
    with torch.no_grad():
        base = model(tensors["sequence"], tensors["lengths"], tensors["temporal_mask"])
        extra = 17
        padded_sequence = torch.cat([
            tensors["sequence"],
            torch.zeros(tensors["sequence"].shape[0], extra,
                        tensors["sequence"].shape[2])], dim=1)
        padded_mask = torch.cat([
            tensors["temporal_mask"],
            torch.zeros(tensors["temporal_mask"].shape[0], extra, dtype=torch.bool)],
            dim=1)
        widened = model(padded_sequence, tensors["lengths"], padded_mask)
        # a single-clip batch must also agree with its row in the multi-clip batch
        single = context.make_fold_batch(0, "validation", branch, probe_clips[:1])
        single_tensors = L.to_torch(single)
        alone = model(single_tensors["sequence"], single_tensors["lengths"],
                     single_tensors["temporal_mask"])
    pad_diff = float((base - widened).abs().max())
    row_diff = float((base[0] - alone[0]).abs().max())
    padding_worst = max(padding_worst, pad_diff, row_diff)
    print(f"  {branch:<8} lengths {tensors['lengths'].tolist()} | "
          f"extra padding {pad_diff:.3e} | batch vs single clip {row_diff:.3e}")
    require(f"{branch}: logits unchanged by {extra} extra padded steps",
            pad_diff < 1e-4, f"{pad_diff:.3e}")
    require(f"{branch}: logits unchanged by batch composition",
            row_diff < 1e-4, f"{row_diff:.3e}")
    require(f"{branch}: lengths stayed on the CPU for packing",
            tensors["lengths"].device.type == "cpu"
            and tensors["lengths"].dtype == torch.int64, "cpu int64")
    del model, batch, tensors
    gc.collect()
require("padding invariance holds for both branches", padding_worst < 1e-4,
        f"max {padding_worst:.3e}")

In [ ]:
# ============================================================
# CELL 5 — Freeze the neural model contract before any training
# ============================================================

NN_MODEL_CONTRACT = {
    "notebook": "11-bilstm-8-fold",
    "nn_policy_version": NN_POLICY_VERSION,
    "model_family": MODEL_FAMILY,
    "upstream": {
        "experiment_id": REQUIRED_EXPERIMENT_ID,
        "loader_contract_id": REQUIRED_LOADER_CONTRACT_ID,
        "standardiser_set_id": REQUIRED_STANDARDISER_SET_ID,
        "time_metadata_id": REQUIRED_TIME_METADATA_ID,
        "loader_module_sha256": REQUIRED_LOADER_MODULE_SHA256,
    },
    "time_base": {
        "policy": "NATIVE FRAMES. Sequences are consumed at their recorded frame rate; "
                  "no resampling, decimation or frame-rate correction is applied to the "
                  "neural input.",
        "why_acceptable": "the ~30 fps (2,365 clips) and ~60 fps (3,635 clips) regimes "
                          "are signer-associated, so a 2.8 s sign spans about 83 frames "
                          "at 30 fps and 166 at 60 fps. This is treated as nuisance "
                          "variance rather than an unseen domain shift because every "
                          "fold's 12 training signers span both regimes: at least 3 of "
                          "the 5 thirty-only and 6 of the 8 sixty-only signers always "
                          "remain in training.",
        "residual_risk": "the network may learn regime-specific temporal templates, and "
                         "because regime tracks signer, a test signer's regime partly "
                         "selects which template applies. This is not leakage - regime "
                         "is not the label - but it adds variance and may interact with "
                         "per-signer difficulty. It must be reported.",
        "contrast_with_random_forest": "the Random Forest branch expressed motion in "
                                       "physical time by scaling with fps. The neural "
                                       "branch does not, so the two model families "
                                       "consume different time bases. Notebook 13 must "
                                       "state this when comparing them.",
        "predeclared_sensitivity": "a common-time analysis - duration-derived uniform "
                                   "sampling through the frozen uniform_indices "
                                   "mechanism, index selection only, no interpolation - "
                                   "is PRE-DECLARED here as a secondary sensitivity "
                                   "analysis and is deliberately NOT executed. Its exact "
                                   "method, including the target rate, is NOT yet frozen "
                                   "and must be frozen before execution. It can never "
                                   "replace this primary native-frame result.",
        "fps_is_never_an_input": True,
    },
    "sequence_policy": {
        "source": "notebook 8 scope-safe API only",
        "max_frames": MAX_FRAMES,
        "at_or_below_max": "every original frame is kept",
        "above_max": "deterministic uniform indices from rint(linspace(0, T-1, 256)), "
                     "strictly increasing, first and last frame retained",
        "forbidden": ["interpolation", "frame repetition", "head or tail truncation",
                      "padding to a fixed length"],
        "padding": "dynamic per batch to the batch maximum, applied after "
                   "standardisation",
        "masks": "temporal padding mask is distinct from the four MediaPipe presence "
                 "masks; presence masks are never z-scored and stay binary",
        "lengths": "int64 on the CPU, as pack_padded_sequence requires",
        "clips_above_max_frames": int((FRAMES > MAX_FRAMES).sum()),
    },
    "standardisation": {
        "source": "frozen notebook 8 standardiser set",
        "primary_folds": "fitted on the 12 training signers of the fold only",
        "control": "fitted on the 4,500 control-training clips only",
        "refitted_here": False,
        "shared_tail_identical_across_branches": True,
    },
    "architecture": {
        "projection": f"Linear(D -> {PROJECTION_DIM}) + LayerNorm + ReLU + "
                      f"Dropout({PROJECTION_DROPOUT})",
        "recurrent": f"{NUM_LAYERS}-layer BiLSTM, hidden {HIDDEN_SIZE} per direction, "
                     f"inter-layer dropout {RECURRENT_DROPOUT}, packed sequences",
        "pooling": f"{POOLING} over true lengths -> {4 * HIDDEN_SIZE}",
        "head": f"Dropout({HEAD_DROPOUT}) -> Linear({4 * HIDDEN_SIZE} -> {N_LABELS})",
        "identical_across_branches_except": "the input projection, whose shape follows "
                                            "the branch feature width",
        "parameters": PARAMETERS,
        "projection_parameter_difference": int(projection_difference),
        "total_parameter_difference": int(total_difference),
        "fairness_statement": "recurrent core matched, total parameter count not "
                              "identical. The recurrent core and the classifier head use "
                              "MATCHED INITIALISATION - their parameters begin "
                              "bit-identically in both branches - while the "
                              "shape-dependent input projections cannot be identical and "
                              "are not made so. The full branch carries 288,768 "
                              "additional input-projection weights and this must be "
                              "stated wherever the branches are compared.",
        "matched_total_parameter_variant": "not run; would change the architecture "
                                           "rather than the representation, so it is "
                                           "at most a secondary sensitivity",
    },
    "training": {
        "loss": f"cross-entropy, label smoothing {LABEL_SMOOTHING}",
        "optimiser": f"AdamW, lr {LEARNING_RATE}, weight decay {WEIGHT_DECAY}",
        "scheduler": dict(SCHEDULER),
        "grad_clip_norm": GRAD_CLIP_NORM,
        "batch_size": BATCH_SIZE, "eval_batch_size": EVAL_BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "early_stopping": {"metric": EARLY_STOPPING_METRIC,
                           "patience": EARLY_STOPPING_PATIENCE,
                           "mode": "max", "restore_best_weights": True},
        "precision": PRECISION,
        "batch_order": "L.epoch_order(clip_ids, scope_seed, epoch): deterministic, "
                       "identical for both branches within a fold",
        "identical_across_branches": "every hyperparameter above is identical for full "
                                     "and reduced within this model family",
        "bigru_may_differ": "BiLSTM and BiGRU may carry different frozen "
                            "hyperparameters; what must match is full versus reduced "
                            "within one model family",
    },
    "tuning_policy": {
        "architecture_search": "none. Every value above was pre-specified before any "
                               "training run.",
        "validation_role": "No hyperparameter tuning is performed. Validation macro-F1 drives the pre-specified learning-rate scheduler and selects the stopping epoch; it does not select architecture or hyperparameters.",
        "why_no_tuning": "the rotation design makes every signer some fold's test "
                         "signer, so there is no tuning set that is clean for all "
                         "folds. Tuning on fold 0's validation signers would make fold "
                         "1 optimistic, because they are fold 1's test signers.",
        "test_policy": "each scope's test signers are predicted exactly once, after "
                       "best-weight restoration. No test metric influences any choice.",
    },
    "initialisation": {
        "policy": "component-wise deterministic initialisation. Each component is seeded "
                  "independently, so the recurrent core and the classifier head begin "
                  "bit-identically in both branches of a scope despite the differently "
                  "shaped input projections consuming different numbers of RNG draws.",
        "offsets": dict(INIT_OFFSETS),
        "offsets_note": "arbitrary but frozen, and identical for both branches",
        "method": "each submodule's own reset_parameters() is re-run under the component "
                  "seed, so PyTorch's default initialisation is reproduced exactly rather "
                  "than replaced",
        "device_independent": "parameters are initialised on the CPU before transfer, so "
                              "initialisation does not depend on the accelerator",
        "training_rng": f"reset to scope_seed + {INIT_OFFSETS['training']} after "
                        "construction, so dropout and batch shuffling begin from the same "
                        "state in both branches",
        "dropout_alignment": "every dropout mask has the same shape in both branches, "
                             "because dropout is applied after the projection to "
                             f"{PROJECTION_DIM} dimensions",
        "projection_not_matched": "the projection matrices have different shapes and are "
                                  "deliberately NOT made identical",
        "verified": ["every recurrent state_dict tensor bit-identical across branches",
                     "every head state_dict tensor bit-identical across branches",
                     "recurrent and head parameter counts identical",
                     "projection shapes correctly different",
                     "total parameter difference exactly 288,768"],
    },
    "seeds": {
        "fold": f"random_state = {NN_FOLD_SEED_BASE} + fold",
        "control": f"random_state = {NN_CONTROL_SEED}",
        "shared_across_branches": "Full and Reduced use the same seed within each fold. "
                                  "Combined with component-wise initialisation this makes "
                                  "the recurrent core and head start bit-identically and "
                                  "the training RNG start from the same state, with "
                                  "identical batch order. The projection parameters "
                                  "cannot be identical because their shapes differ.",
        "determinism": DETERMINISM_NOTE,
    },
    "probability_class_order": "columns are class ids 0..199 ascending; softmax over the "
                              "200 logits",
    "predictions_from": "argmax over the softmax probabilities",
    "metrics": {
        "per_fold": ["accuracy", "macro_f1", "weighted_f1", "balanced_accuracy",
                     f"top_{TOP_K}_accuracy", "per_test_signer_accuracy",
                     "signer_macro_accuracy"],
        "pooled_oof": ["accuracy", "macro_f1", "weighted_f1", "balanced_accuracy",
                       f"top_{TOP_K}_accuracy", "signer_macro_accuracy",
                       "per_signer_accuracy", "per_class_recall",
                       "confusion_matrix_200x200"],
        "validation": "saved separately and labelled DIAGNOSTIC ONLY. Validation "
                      "macro-F1 drives the pre-specified learning-rate scheduler and "
                      "selects the stopping epoch; it does not select architecture or "
                      "hyperparameters.",
        "deferred_to_notebook_13": ["paired sign-flip test", "confidence intervals",
                                    "any equivalence or significance claim",
                                    "cross-model comparison"],
    },
    "oof_policy": {
        "definition": "each of the 6,000 clips receives exactly one prediction, from the "
                      "fold in which its signer is a TEST signer",
        "probability_dtype": "float32",
    },
    "control_policy": {
        "runs": "exactly one signer-overlap control per branch",
        "splits": dict(CONTROL_SPLIT_SIZES),
        "label": "INTENTIONAL SIGNER-OVERLAP CONTROL",
        "separation": "never mixed with the 6,000 signer-independent out-of-fold "
                      "results",
        "purpose": "measures the signer-overlap performance gap for this model family so "
                   "it can be compared with the Random Forest gap",
    },
    "resource_gate": {
        "max_gpu_memory_gib": MAX_GPU_MEMORY_GIB,
        "probe": "a short training and evaluation pass on fold 0 TRAIN and VALIDATION "
                 "rows only, per branch",
        "action_on_breach": "STOP before the primary training loop. No test prediction "
                            "is generated, so the architecture or batch size can be "
                            "reconsidered for resource reasons without any evaluation "
                            "result having been seen.",
        "automatic_adjustment": False,
        "runtime_is_reported_not_gated": "projected wall clock is reported only, because "
                                         "resume across sessions handles a long run",
    },
    "resume_rules": {
        "scopes": len(ALL_SCOPES),
        "overwrite_default": OVERWRITE,
        "artefact_written": "immediately after each scope completes, atomically via a "
                           "temporary file and rename",
        "cross_session": "completed artefacts are also read from attached inputs, so a "
                         "run may span Kaggle sessions",
        "skip_requires": ["neural model contract id matches",
                          "upstream provenance matches", "expected row counts match",
                          "prediction digest reproduces from the saved arrays",
                          "probability shape and class order valid"],
        "on_mismatch": "recompute the scope; stale partial output is never silently "
                       "reused",
        "session_budget_seconds": SESSION_TIME_BUDGET_SECONDS,
    },
    "forbidden": ["regenerating folds, standardisers or the sequence policy",
                  "tuning on validation beyond the stopping epoch and the schedule",
                  "any use of test predictions in a decision",
                  "retraining on train + validation",
                  "mixing control and out-of-fold predictions",
                  "claiming identical total parameter counts",
                  "resampling or frame-rate correcting the neural input in this "
                  "primary notebook"],
}
NN_MODEL_CONTRACT_ID = sha16(json.dumps(NN_MODEL_CONTRACT, sort_keys=True,
                                        separators=(",", ":"), default=str))

print("NEURAL MODEL CONTRACT")
print(f"  nn model contract id : {NN_MODEL_CONTRACT_ID}")
print(f"  model family         : {MODEL_FAMILY}")
print(f"  time base            : native frames; common-time sensitivity pre-declared,")
print(f"                         exact method not yet frozen")
print(f"  tuning               : none; validation macro-F1 drives the frozen "
      f"scheduler and the stopping epoch only")

blob = json.dumps(NN_MODEL_CONTRACT, default=str)
require("the contract carries no session path", "/kaggle" not in blob, "stable")
require("the contract carries no timestamp", "generated_at" not in blob, "stable")
require("the contract records the native-frame time base",
        NN_MODEL_CONTRACT["time_base"]["policy"].startswith("NATIVE FRAMES"), "recorded")
require("the contract pre-declares the common-time sensitivity as non-replacing",
        "never replace" in NN_MODEL_CONTRACT["time_base"]["predeclared_sensitivity"]
        and "must be frozen before execution"
        in NN_MODEL_CONTRACT["time_base"]["predeclared_sensitivity"],
        "method not yet frozen")
require("the contract records the RF/neural time-base contrast",
        "different time bases"
        in NN_MODEL_CONTRACT["time_base"]["contrast_with_random_forest"], "recorded")
require("the contract states the fairness wording correctly",
        "recurrent core matched, total parameter count not identical"
        in NN_MODEL_CONTRACT["architecture"]["fairness_statement"], "recorded")
# Scoped deliberately: the "forbidden" list legitimately contains the phrase
# "claiming identical total parameter counts", so it is excluded from the search.
claims = json.dumps({k: v for k, v in NN_MODEL_CONTRACT.items() if k != "forbidden"},
                    default=str)
false_claims = ["totals are identical", "identical total parameter count",
                "identical parameter counts", "same total parameter"]
require("the contract makes no false claim of identical totals",
        not any(phrase in claims for phrase in false_claims)
        and PARAMETERS["full"]["total"] != PARAMETERS["reduced"]["total"],
        f"{PARAMETERS['full']['total']:,} vs {PARAMETERS['reduced']['total']:,}")
require("the contract forbids claiming identical totals",
        "claiming identical total parameter counts" in NN_MODEL_CONTRACT["forbidden"],
        "recorded as forbidden")
require("the contract records that no tuning occurred",
        NN_MODEL_CONTRACT["tuning_policy"]["architecture_search"].startswith("none"),
        "pre-specified")
require("the contract states the validation role accurately",
        "drives the pre-specified learning-rate scheduler"
        in NN_MODEL_CONTRACT["tuning_policy"]["validation_role"]
        and "does not select architecture or hyperparameters"
        in NN_MODEL_CONTRACT["tuning_policy"]["validation_role"],
        "scheduler and stopping epoch, nothing else")
require("no wording claims validation affects nothing but the epoch",
        "stopping epoch and nothing else" not in blob, "corrected")
require("the contract records the test-once policy",
        "exactly once" in NN_MODEL_CONTRACT["tuning_policy"]["test_policy"], "recorded")
require("the contract acknowledges the projections cannot align",
        "cannot be identical" in NN_MODEL_CONTRACT["seeds"]["shared_across_branches"],
        "different projection shapes")
require("the contract records the frozen initialisation offsets",
        NN_MODEL_CONTRACT["initialisation"]["offsets"] == INIT_OFFSETS
        and len(set(INIT_OFFSETS.values())) == len(INIT_OFFSETS), INIT_OFFSETS)
require("the contract records matched recurrent and head initialisation",
        "bit-identically" in NN_MODEL_CONTRACT["initialisation"]["policy"]
        and "MATCHED INITIALISATION"
        in NN_MODEL_CONTRACT["architecture"]["fairness_statement"], "recorded")
require("the contract records the GPU resource gate",
        NN_MODEL_CONTRACT["resource_gate"]["max_gpu_memory_gib"] == MAX_GPU_MEMORY_GIB
        and NN_MODEL_CONTRACT["resource_gate"]["automatic_adjustment"] is False,
        f"{MAX_GPU_MEMORY_GIB} GiB, no automatic change")

atomic_write(OUT["contract"], lambda h: h.write(
    json.dumps(NN_MODEL_CONTRACT, indent=2, sort_keys=True, default=str).encode()))
reloaded = json.loads(OUT["contract"].read_text(encoding="utf-8"))
require("the neural model contract id reproduces from the file on disk",
        sha16(json.dumps(reloaded, sort_keys=True, separators=(",", ":"),
                         default=str)) == NN_MODEL_CONTRACT_ID, NN_MODEL_CONTRACT_ID)
print(f"  wrote {OUT['contract'].name}  {OUT['contract'].stat().st_size:,} bytes")

In [ ]:
# ============================================================
# CELL 6 — Training helpers, metrics, resume artefacts, batch fidelity
# ============================================================


def batches_from_cache(cache: Dict[str, np.ndarray], clip_ids: Sequence[str],
                       batch_size: int, scope_tag: str, shuffle: bool = False,
                       seed: int = 0, epoch: int = 0):
    """Assemble batches with the frozen collate, so padding and masks are the
    loader's own semantics rather than a reimplementation."""
    order = (L.epoch_order(clip_ids, seed, epoch) if shuffle else sorted(clip_ids))
    for start in range(0, len(order), batch_size):
        chunk = order[start:start + batch_size]
        yield L.collate_variable_length(
            [cache[c] for c in chunk],
            [int(LABEL_OF[c]) for c in chunk], list(chunk),
            [str(SIGNER_OF[c]) for c in chunk], scope=scope_tag)


def to_device(batch: Dict[str, Any]) -> Dict[str, Any]:
    tensors = L.to_torch(batch, device=DEVICE)
    return tensors


def top_k_accuracy(proba: np.ndarray, y_true: np.ndarray, k: int = TOP_K) -> float:
    top = np.argpartition(-proba, kth=k - 1, axis=1)[:, :k]
    return float(np.mean([y_true[i] in top[i] for i in range(y_true.size)]))


def scope_metrics(y_true: np.ndarray, y_pred: np.ndarray, proba: np.ndarray,
                  signers: np.ndarray) -> Dict[str, Any]:
    per_signer = {}
    for signer in sorted(set(signers.tolist()), key=lambda v: int(str(v).lstrip("p"))):
        mask = signers == signer
        per_signer[str(signer)] = float(accuracy_score(y_true[mask], y_pred[mask]))
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted",
                                      zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        f"top{TOP_K}_accuracy": top_k_accuracy(proba, y_true),
        "signer_macro_accuracy": float(np.mean(list(per_signer.values()))),
        "per_signer_accuracy": per_signer, "n": int(y_true.size),
    }


def probabilities_valid(proba: np.ndarray, rows: int) -> bool:
    return (proba.shape == (rows, N_LABELS) and proba.dtype == np.float32
            and bool(np.isfinite(proba).all()) and float(proba.min()) >= 0.0
            and float(np.abs(proba.sum(axis=1) - 1.0).max()) < 1e-3)


@torch.no_grad()
def evaluate(model: nn.Module, batch_iterable) -> Dict[str, np.ndarray]:
    """Softmax probabilities for a split. Returns arrays ordered as iterated."""
    model.eval()
    clip_ids, signers, y_true, chunks = [], [], [], []
    for batch in batch_iterable:
        tensors = to_device(batch)
        logits = model(tensors["sequence"], tensors["lengths"],
                       tensors["temporal_mask"])
        chunks.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
        clip_ids.extend(batch["clip_id"])
        signers.extend(batch["signer_id"])
        y_true.append(batch["label_id"])
    proba = np.concatenate(chunks).astype(np.float32)
    return {"clip_id": np.array(clip_ids, dtype=object).astype(str),
            "signer_id": np.array(signers, dtype=object).astype(str),
            "y_true": np.concatenate(y_true).astype(np.int64),
            "y_pred": np.argmax(proba, axis=1).astype(np.int64), "proba": proba}


def train_one_epoch(model: nn.Module, optimiser, criterion, batch_iterable) -> float:
    model.train()
    total_loss, total_items = 0.0, 0
    for batch in batch_iterable:
        tensors = to_device(batch)
        optimiser.zero_grad(set_to_none=True)
        logits = model(tensors["sequence"], tensors["lengths"],
                      tensors["temporal_mask"])
        loss = criterion(logits, tensors["label_id"])
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimiser.step()
        items = int(tensors["label_id"].shape[0])
        total_loss += float(loss.detach()) * items
        total_items += items
    return total_loss / max(total_items, 1)


# ---- resume artefacts ------------------------------------------------------
SCOPE_PROVENANCE = {
    "experiment_id": REQUIRED_EXPERIMENT_ID,
    "loader_contract_id": REQUIRED_LOADER_CONTRACT_ID,
    "standardiser_set_id": REQUIRED_STANDARDISER_SET_ID,
    "time_metadata_id": REQUIRED_TIME_METADATA_ID,
    "nn_model_contract_id": NN_MODEL_CONTRACT_ID,
}
# Cross-session resume: completed artefacts may live in an attached earlier version.
RESUME_DIRS = [SCOPE_DIR] + sorted(
    {p.parent for p in Path(SEARCH_ROOT).glob("**/nn_scopes/nn_scope_*.npz")})
if len(RESUME_DIRS) > 1:
    print(f"resume sources: {[str(d) for d in RESUME_DIRS]}")


def scope_filename(branch: str, scope: Any) -> str:
    return f"nn_scope_{scope_label(branch, scope)}.npz"


def save_scope(branch: str, scope: Any, payload: Dict[str, Any]) -> Path:
    path = SCOPE_DIR / scope_filename(branch, scope)
    arrays = {k: v for k, v in payload.items() if isinstance(v, np.ndarray)}
    meta = {k: v for k, v in payload.items() if not isinstance(v, np.ndarray)}
    arrays["__meta__"] = np.array([json.dumps(meta, sort_keys=True, default=str)])
    temporary = path.parent / (path.name + ".tmp")
    with open(temporary, "wb") as handle:
        np.savez_compressed(handle, **arrays)
    with np.load(temporary, allow_pickle=False) as check:
        reloaded_meta = json.loads(str(check["__meta__"][0]))
        ok = (reloaded_meta["prediction_digest"]
              == array_digest(check["y_pred"], check["proba"])
              and probabilities_valid(check["proba"], int(check["y_true"].size)))
    if not ok:
        os.unlink(temporary)
        raise RuntimeError(f"{path.name} failed post-write validation; not published")
    os.replace(temporary, path)
    return path


def load_scope(branch: str, scope: Any) -> Optional[Dict[str, Any]]:
    if OVERWRITE:
        return None
    for directory in RESUME_DIRS:
        path = directory / scope_filename(branch, scope)
        if not path.exists():
            continue
        try:
            with np.load(path, allow_pickle=False) as archive:
                meta = json.loads(str(archive["__meta__"][0]))
                arrays = {k: np.array(archive[k]) for k in archive.files
                          if k != "__meta__"}
        except Exception as error:
            print(f"    resume rejected at {path.parent.name} "
                  f"({type(error).__name__})")
            continue
        clips = scope_clips(scope)
        reasons = []
        if {k: meta.get(k) for k in SCOPE_PROVENANCE} != SCOPE_PROVENANCE:
            reasons.append("provenance or contract id")
        if meta.get("branch") != branch or str(meta.get("scope")) != str(scope):
            reasons.append("scope label")
        if int(meta.get("n_test", -1)) != len(clips["test"]):
            reasons.append("test rows")
        if int(meta.get("n_train", -1)) != len(clips["train"]):
            reasons.append("train rows")
        if meta.get("prediction_digest") != array_digest(arrays["y_pred"],
                                                         arrays["proba"]):
            reasons.append("prediction digest")
        if not probabilities_valid(arrays["proba"], len(clips["test"])):
            reasons.append("probability validity")
        if reasons:
            print(f"    resume rejected at {path.parent.name} "
                  f"({', '.join(reasons)})")
            continue
        if directory != SCOPE_DIR:          # copy forward so outputs are complete
            import shutil
            shutil.copy2(path, SCOPE_DIR / path.name)
        return {**meta, **arrays}
    return None


# ---- batch fidelity: the cache must reproduce the loader's own batches -----
print("BATCH FIDELITY (cached batching vs the loader's scope-safe batching)")
fidelity_worst = 0
for branch in BRANCHES:
    probe_clips = sorted(context.clip_ids(0, "validation"))[:8]
    reference = context.make_fold_batch(0, "validation", branch, probe_clips)
    cache = {clip_id: np.ascontiguousarray(
        reference["sequence"][i, :int(reference["lengths"][i])], dtype=np.float32)
        for i, clip_id in enumerate(reference["clip_id"])}
    rebuilt = next(batches_from_cache(cache, probe_clips, len(probe_clips),
                                      scope_tag="fidelity"))
    same = (rebuilt["sequence"].tobytes() == reference["sequence"].tobytes()
            and np.array_equal(rebuilt["lengths"], reference["lengths"])
            and np.array_equal(rebuilt["temporal_mask"], reference["temporal_mask"])
            and np.array_equal(rebuilt["label_id"], reference["label_id"])
            and rebuilt["clip_id"] == reference["clip_id"])
    fidelity_worst += 0 if same else 1
    print(f"  {branch:<8} {len(probe_clips)} clips, padded to "
          f"{reference['padded_length']} | byte-identical: {same}")
    require(f"{branch}: cached batching is byte-identical to the loader's batching",
            same, "sequence, lengths, mask, labels and order")
    start, end = MASK_SLICE[branch]
    require(f"{branch}: presence masks remain strictly binary after standardisation",
            bool(np.isin(reference["sequence"][:, :, start:end], (0.0, 1.0)).all()),
            "0/1 preserved")
    require(f"{branch}: temporal mask sums to the true lengths",
            int(reference["temporal_mask"].sum()) == int(reference["lengths"].sum()),
            f"{int(reference['lengths'].sum())} real frames")
    del reference, cache, rebuilt
    gc.collect()
require("cached batching matches the loader for both branches", fidelity_worst == 0,
        "no reimplementation drift")

completed_now = sum(1 for b, s in ALL_SCOPES if load_scope(b, s) is not None)
print(f"\nscopes to run    : {len(ALL_SCOPES)}")
print(f"already complete : {completed_now} (each re-verified before reuse)")

In [ ]:
# ============================================================
# CELL 7 — Resource preflight and GPU memory gate
# ============================================================
# Fold 0 train and validation rows only. Measures one training epoch and one
# validation pass per branch, then projects. Memory is a hard gate; wall clock is
# reported only, because resume across sessions handles a long run. Nothing is
# adjusted automatically and no test data is touched.

require("a CUDA device is available",
        torch.cuda.is_available(),
        ENVIRONMENT["gpu"] or "no GPU: enable the GPU accelerator and rerun")
print(f"PREFLIGHT  fold 0, one epoch per branch, batch {BATCH_SIZE}")
PREFLIGHT: List[Dict[str, Any]] = []
for branch in BRANCHES:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    started = time.time()
    train_cache = build_cache(branch, 0, "train")
    validation_cache = build_cache(branch, 0, "validation")
    cache_seconds = time.time() - started
    cache_gib = (sum(a.nbytes for a in train_cache.values())
                 + sum(a.nbytes for a in validation_cache.values())) / (1024 ** 3)

    model = build_model(BRANCH_WIDTH[branch], scope_seed(0)).to(DEVICE)
    optimiser = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                                  weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    started = time.time()
    train_loss = train_one_epoch(model, optimiser, criterion, batches_from_cache(
        train_cache, context.clip_ids(0, "train"), BATCH_SIZE,
        scope_tag=f"preflight:{branch}", shuffle=True, seed=scope_seed(0), epoch=0))
    epoch_seconds = time.time() - started

    started = time.time()
    predictions = evaluate(model, batches_from_cache(
        validation_cache, context.clip_ids(0, "validation"), EVAL_BATCH_SIZE,
        scope_tag=f"preflight:{branch}"))
    validation_seconds = time.time() - started
    peak = gpu_peak_gib()

    projected_scope = cache_seconds + MAX_EPOCHS * (epoch_seconds + validation_seconds)
    PREFLIGHT.append({
        "branch": branch, "cache_seconds": round(cache_seconds, 1),
        "cache_gib": round(cache_gib, 3),
        "train_epoch_seconds": round(epoch_seconds, 1),
        "validation_seconds": round(validation_seconds, 1),
        "train_loss_first_epoch": round(train_loss, 4),
        "peak_gpu_gib": round(peak, 3),
        "projected_scope_seconds_max_epochs": round(projected_scope, 1),
        "parameters": PARAMETERS[branch]["total"],
    })
    print(f"  {branch:<8} cache {cache_gib:.2f} GiB in {cache_seconds:.0f}s | "
          f"epoch {epoch_seconds:.1f}s | validation {validation_seconds:.1f}s | "
          f"peak GPU {peak:.2f} GiB")
    print(f"           worst-case scope at {MAX_EPOCHS} epochs "
          f"~{projected_scope / 60:.0f} min (early stopping usually shortens this)")
    require(f"{branch} preflight probabilities are valid",
            probabilities_valid(predictions["proba"],
                               len(context.clip_ids(0, "validation"))),
            f"{predictions['proba'].shape}")
    require(f"{branch} the first epoch produced a finite loss",
            np.isfinite(train_loss) and train_loss > 0, f"{train_loss:.4f}")
    del model, optimiser, criterion, train_cache, validation_cache, predictions
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

preflight_table = pd.DataFrame(PREFLIGHT)
worst_gpu = float(preflight_table["peak_gpu_gib"].max())
projected_all = float(preflight_table["projected_scope_seconds_max_epochs"].sum()) * (
    N_FOLDS + 1)

print(f"\nRESOURCE GATE  ceiling {MAX_GPU_MEMORY_GIB} GiB of GPU memory")
for row in PREFLIGHT:
    print(f"  {row['branch']:<8} peak {row['peak_gpu_gib']:.2f} GiB | "
          f"{row['parameters']:,} parameters | RAM cache {row['cache_gib']:.2f} GiB")
print(f"  worst peak GPU       : {worst_gpu:.2f} GiB")
print(f"  projected all scopes : ~{projected_all / 3600:.1f} h at {MAX_EPOCHS} epochs "
      f"(reported, NOT gated: resume across sessions handles it)")

PREFLIGHT_REPORT = {
    "notebook": "11-bilstm-8-fold", "stage": "resource_gate",
    "nn_model_contract_id": NN_MODEL_CONTRACT_ID, "upstream": dict(SCOPE_PROVENANCE),
    "max_gpu_memory_gib": MAX_GPU_MEMORY_GIB,
    "worst_peak_gpu_gib": round(worst_gpu, 3),
    "projected_all_scopes_hours": round(projected_all / 3600, 2),
    "session_budget_hours": round(SESSION_TIME_BUDGET_SECONDS / 3600, 2),
    "per_branch": PREFLIGHT, "device": str(DEVICE),
    "probe_scope": "fold 0 TRAIN and VALIDATION rows only; no test data was touched",
    "verdict": "PASS" if worst_gpu <= MAX_GPU_MEMORY_GIB else "STOP",
}
atomic_write(OUT["preflight_report"], lambda h: h.write(
    json.dumps(PREFLIGHT_REPORT, indent=2, default=str).encode()))
print(f"  wrote {OUT['preflight_report'].name}")

if worst_gpu > MAX_GPU_MEMORY_GIB:
    record("resource gate: peak GPU memory within the ceiling", False,
           f"{worst_gpu:.2f} GiB > {MAX_GPU_MEMORY_GIB:.2f} GiB")
    atomic_write(OUT["report_json"], lambda h: h.write(json.dumps({
        "notebook": "11-bilstm-8-fold", "verdict": "STOPPED_AT_RESOURCE_GATE",
        "stage": "resource_gate", "nn_model_contract_id": NN_MODEL_CONTRACT_ID,
        "preflight": PREFLIGHT_REPORT, "checks": CHECKS,
        "runtime_seconds": round(time.time() - STARTED, 1)},
        indent=2, default=str).encode()))
    atomic_write(OUT["report_csv"], lambda h: pd.DataFrame(CHECKS)[
        ["check", "status", "passed", "detail"]].to_csv(h, index=False))
    print("\n" + "=" * 78)
    print("STOPPED AT THE RESOURCE GATE - NO MODEL TRAINED, NO TEST PREDICTION MADE")
    print("=" * 78)
    for row in PREFLIGHT:
        print(f"  {row['branch']:<8} peak GPU {row['peak_gpu_gib']:.2f} GiB")
    print(f"  worst {worst_gpu:.2f} GiB exceeds the frozen ceiling "
          f"{MAX_GPU_MEMORY_GIB:.2f} GiB.")
    print("  Everything above came from fold 0 train and validation rows only. No test")
    print("  prediction exists, so the batch size or architecture may still be")
    print("  reconsidered for resource reasons without any evaluation result having")
    print("  been seen. This notebook adjusts nothing. Decide, record the change as a")
    print("  frozen policy amendment, and rerun.")
    print("=" * 78)
    raise RuntimeError(
        f"Resource gate STOP: peak GPU {worst_gpu:.2f} GiB exceeds the frozen ceiling "
        f"{MAX_GPU_MEMORY_GIB:.2f} GiB. No training was started and no test prediction "
        "was generated.")

require("resource gate: peak GPU memory within the ceiling", True,
        f"{worst_gpu:.2f} GiB <= {MAX_GPU_MEMORY_GIB:.2f} GiB")
require("preflight used fold 0 train and validation rows only", True,
        "no test row was touched")
print("\n  Gate PASSED. Timing and memory only; no frozen hyperparameter was revised.")

In [ ]:
# ============================================================
# CELL 8 — Train the 18 scopes: 8 folds x 2 branches, plus 1 control per branch
# ============================================================
# One model on the GPU at a time. No hyperparameter tuning is performed: validation
# macro-F1 drives the pre-specified learning-rate scheduler and selects the stopping
# epoch, and selects no architecture or hyperparameter. Test is predicted once, after
# best weights are restored. A session stops cleanly when the time budget is reached; rerun with this
# version's output attached to continue.

SCOPE_RESULTS: Dict[Tuple[str, Any], Dict[str, Any]] = {}
EFFICIENCY: List[Dict[str, Any]] = []
HISTORY: List[Dict[str, Any]] = []
SCOPE_PROJECTION = {row["branch"]: row["projected_scope_seconds_max_epochs"]
                    for row in PREFLIGHT}
stopped_for_budget = False

for branch, scope in ALL_SCOPES:
    label = scope_label(branch, scope)
    is_control = scope == "control"

    resumed = load_scope(branch, scope)
    if resumed is not None:
        SCOPE_RESULTS[(branch, scope)] = resumed
        EFFICIENCY.append(resumed["efficiency"])
        HISTORY.extend(resumed["history"])
        print(f"\n{label.upper()}  resumed | test accuracy "
              f"{resumed['metrics_test']['accuracy']:.4f} | best epoch "
              f"{resumed['best_epoch']}")
        record(f"{label}: resumed artefact passed every integrity check", True,
               resumed["prediction_digest"])
        continue

    elapsed = time.time() - STARTED
    projected = SCOPE_PROJECTION[branch] * SCOPE_TIME_SAFETY_FACTOR
    if elapsed + projected > SESSION_TIME_BUDGET_SECONDS:
        print(f"\nSESSION BUDGET REACHED before {label}: {elapsed / 3600:.2f} h used, "
              f"next scope projected {projected / 3600:.2f} h, budget "
              f"{SESSION_TIME_BUDGET_SECONDS / 3600:.2f} h")
        stopped_for_budget = True
        break

    print(f"\n{'=' * 74}\n{label.upper()}"
          + ("   [INTENTIONAL SIGNER-OVERLAP CONTROL]" if is_control else "")
          + f"\n{'=' * 74}")
    clips = scope_clips(scope)
    if not is_control:
        definition = context.folds[scope]
        require(f"{label}: train / validation / test signers disjoint",
                not (set(definition["train"]) & set(definition["validation"]))
                and not (set(definition["train"]) & set(definition["test"]))
                and not (set(definition["validation"]) & set(definition["test"])),
                f"{len(definition['train'])}/2/2")
        require(f"{label}: training signers are exactly the frozen 12",
                set(SIGNER_OF.loc[clips["train"]]) == set(definition["train"]),
                len(definition["train"]))
        require(f"{label}: all 200 classes present in training",
                LABEL_OF.loc[clips["train"]].nunique() == N_LABELS, N_LABELS)
    else:
        require(f"{label}: signer overlap is present by design",
                len(set(SIGNER_OF.loc[clips["train"]])
                    & set(SIGNER_OF.loc[clips["test"]])) == N_SIGNERS,
                f"{N_SIGNERS} of {N_SIGNERS}")
    require(f"{label}: no validation or test clip appears in training",
            not (set(clips["train"]) & set(clips["validation"]))
            and not (set(clips["train"]) & set(clips["test"])), "disjoint")

    seed = scope_seed(scope)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    started = time.time()
    train_cache = build_cache(branch, scope, "train")
    validation_cache = build_cache(branch, scope, "validation")
    cache_seconds = time.time() - started

    model = build_model(BRANCH_WIDTH[branch], seed).to(DEVICE)
    optimiser = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                                  weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimiser, mode=SCHEDULER["mode"], factor=SCHEDULER["factor"],
        patience=SCHEDULER["patience"])
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    best_metric, best_epoch, best_state, patience_used = -1.0, -1, None, 0
    scope_history: List[Dict[str, Any]] = []
    training_started = time.time()
    for epoch in range(MAX_EPOCHS):
        train_loss = train_one_epoch(model, optimiser, criterion, batches_from_cache(
            train_cache, clips["train"], BATCH_SIZE, scope_tag=f"{label}:train",
            shuffle=True, seed=seed, epoch=epoch))
        predictions = evaluate(model, batches_from_cache(
            validation_cache, clips["validation"], EVAL_BATCH_SIZE,
            scope_tag=f"{label}:validation"))
        macro_f1 = float(f1_score(predictions["y_true"], predictions["y_pred"],
                                  average="macro", zero_division=0))
        accuracy = float(accuracy_score(predictions["y_true"], predictions["y_pred"]))
        scheduler.step(macro_f1)
        scope_history.append({
            "branch": branch, "scope": str(scope), "epoch": epoch,
            "train_loss": round(train_loss, 5),
            "validation_macro_f1": round(macro_f1, 5),
            "validation_accuracy": round(accuracy, 5),
            "learning_rate": float(optimiser.param_groups[0]["lr"]),
            "seconds": round(time.time() - training_started, 1),
        })
        improved = macro_f1 > best_metric
        if improved:
            best_metric, best_epoch, patience_used = macro_f1, epoch, 0
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        else:
            patience_used += 1
        if epoch % 5 == 0 or improved or patience_used >= EARLY_STOPPING_PATIENCE:
            print(f"  epoch {epoch:>2} | loss {train_loss:.4f} | val macro-F1 "
                  f"{macro_f1:.4f} | val acc {accuracy:.4f} | lr "
                  f"{optimiser.param_groups[0]['lr']:.2e}"
                  + ("  <- best" if improved else f"  patience {patience_used}"))
        if patience_used >= EARLY_STOPPING_PATIENCE:
            print(f"  early stopping at epoch {epoch} (best epoch {best_epoch})")
            break
    training_seconds = time.time() - training_started

    require(f"{label}: a best epoch was selected on validation macro-F1",
            best_epoch >= 0 and best_state is not None, f"epoch {best_epoch}")
    model.load_state_dict(best_state)
    validation_predictions = evaluate(model, batches_from_cache(
        validation_cache, clips["validation"], EVAL_BATCH_SIZE,
        scope_tag=f"{label}:validation"))
    del train_cache, validation_cache
    gc.collect()

    # ---- the single test evaluation ----------------------------------------
    started = time.time()
    test_iterator = (context.iterate_control_batches("test", branch,
                                                     batch_size=EVAL_BATCH_SIZE)
                     if is_control
                     else context.iterate_fold_batches(scope, "test", branch,
                                                       batch_size=EVAL_BATCH_SIZE))
    test_predictions = evaluate(model, test_iterator)
    test_seconds = time.time() - started
    require(f"{label}: test predicted in a single pass after best-weight restoration",
            test_predictions["y_true"].size == len(clips["test"])
            and len(set(test_predictions["clip_id"].tolist())) == len(clips["test"]),
            f"{test_predictions['y_true'].size} clips, no duplicates")
    require(f"{label}: test probabilities are valid",
            probabilities_valid(test_predictions["proba"], len(clips["test"])),
            f"{test_predictions['proba'].shape}, row sums ~1")
    require(f"{label}: test predictions equal argmax of the probabilities",
            np.array_equal(test_predictions["y_pred"],
                           np.argmax(test_predictions["proba"], axis=1)), "0..199")
    require(f"{label}: test labels match the frozen manifest",
            np.array_equal(test_predictions["y_true"],
                           LABEL_OF.loc[test_predictions["clip_id"]].to_numpy()),
            "aligned")

    metrics_test = scope_metrics(test_predictions["y_true"], test_predictions["y_pred"],
                                test_predictions["proba"],
                                test_predictions["signer_id"])
    metrics_validation = scope_metrics(
        validation_predictions["y_true"], validation_predictions["y_pred"],
        validation_predictions["proba"], validation_predictions["signer_id"])
    print(f"  test accuracy {metrics_test['accuracy']:.4f} | macro F1 "
          f"{metrics_test['macro_f1']:.4f} | top{TOP_K} "
          f"{metrics_test[f'top{TOP_K}_accuracy']:.4f}")
    print(f"  validation accuracy {metrics_validation['accuracy']:.4f}   "
          "[DIAGNOSTIC ONLY]")

    efficiency = {
        "branch": branch, "scope": str(scope), "seed": seed,
        "parameters": PARAMETERS[branch]["total"],
        "projection_parameters": PARAMETERS[branch]["projection"],
        "train_clips": len(clips["train"]), "epochs_run": len(scope_history),
        "best_epoch": int(best_epoch),
        "best_validation_macro_f1": round(best_metric, 5),
        "cache_seconds": round(cache_seconds, 1),
        "train_seconds": round(training_seconds, 1),
        "seconds_per_epoch": round(training_seconds / max(len(scope_history), 1), 2),
        "test_inference_seconds": round(test_seconds, 2),
        "peak_gpu_gib": round(gpu_peak_gib(), 3),
    }
    EFFICIENCY.append(efficiency)
    HISTORY.extend(scope_history)

    payload = {
        **SCOPE_PROVENANCE, "branch": branch, "scope": str(scope), "seed": seed,
        "is_control": bool(is_control), "n_train": len(clips["train"]),
        "n_validation": len(clips["validation"]), "n_test": len(clips["test"]),
        "best_epoch": int(best_epoch), "epochs_run": len(scope_history),
        "clip_id": test_predictions["clip_id"],
        "signer_id": test_predictions["signer_id"],
        "y_true": test_predictions["y_true"], "y_pred": test_predictions["y_pred"],
        "proba": test_predictions["proba"],
        "val_clip_id": validation_predictions["clip_id"],
        "val_y_true": validation_predictions["y_true"],
        "val_y_pred": validation_predictions["y_pred"],
        "val_proba": validation_predictions["proba"],
        "metrics_test": metrics_test, "metrics_validation": metrics_validation,
        "validation_status": "DIAGNOSTIC ONLY", "efficiency": efficiency,
        "history": scope_history,
        "prediction_digest": array_digest(test_predictions["y_pred"],
                                          test_predictions["proba"]),
    }
    written = save_scope(branch, scope, payload)
    SCOPE_RESULTS[(branch, scope)] = payload
    print(f"  wrote {written.name}  {written.stat().st_size / 1e6:.1f} MB | "
          f"{len(scope_history)} epochs in {training_seconds / 60:.1f} min")

    del model, optimiser, scheduler, criterion, best_state
    del test_predictions, validation_predictions
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

PIPELINE_COMPLETE = len(SCOPE_RESULTS) == len(ALL_SCOPES)
print(f"\n{'=' * 74}")
print(f"scopes complete: {len(SCOPE_RESULTS)} of {len(ALL_SCOPES)}")
if PIPELINE_COMPLETE:
    require("every scope produced a result", True, f"{len(ALL_SCOPES)} of {len(ALL_SCOPES)}")
    require("full and reduced used the same seed within every scope",
            all(SCOPE_RESULTS[("full", s)]["seed"] == SCOPE_RESULTS[("reduced", s)]["seed"]
                for s in list(range(N_FOLDS)) + ["control"]), "paired seeds")
    require("seeds follow the frozen policy",
            all(SCOPE_RESULTS[(b, f)]["seed"] == NN_FOLD_SEED_BASE + f
                for b in BRANCHES for f in range(N_FOLDS))
            and all(SCOPE_RESULTS[(b, "control")]["seed"] == NN_CONTROL_SEED
                    for b in BRANCHES),
            f"{NN_FOLD_SEED_BASE} + k, control {NN_CONTROL_SEED}")
    require("every best epoch was selected before the maximum budget or at it",
            all(0 <= SCOPE_RESULTS[k]["best_epoch"] < MAX_EPOCHS for k in SCOPE_RESULTS),
            f"max {MAX_EPOCHS} epochs")
else:
    remaining = [scope_label(b, s) for b, s in ALL_SCOPES
                 if (b, s) not in SCOPE_RESULTS]
    print(f"remaining      : {', '.join(remaining)}")
    print("\n" + "=" * 74)
    print("PARTIAL RUN - RESUME REQUIRED")
    print("=" * 74)
    print("  Completed scopes are saved and verified. Nothing downstream is computed,")
    print("  because pooled out-of-fold results need all 8 folds per branch.")
    print("  To continue: commit this version, attach its output as an extra input, and")
    print("  rerun. Completed scopes will be recognised and skipped.")
    if stopped_for_budget:
        print(f"  Reason: session time budget "
              f"({SESSION_TIME_BUDGET_SECONDS / 3600:.1f} h) would have been exceeded.")
    print("=" * 74)

In [ ]:
# ============================================================
# CELL 9 — Out-of-fold assembly: exactly one prediction per clip per branch
# ============================================================

if not PIPELINE_COMPLETE:
    print("SKIPPED: the run is partial. Pooled out-of-fold results require all 8 folds")
    print("per branch. Attach this version's output and rerun to continue.")
else:
    OOF: Dict[str, Dict[str, np.ndarray]] = {}
    for branch in BRANCHES:
        parts, probability_parts = [], []
        for fold in range(N_FOLDS):
            scope = SCOPE_RESULTS[(branch, fold)]
            parts.append(pd.DataFrame({
                "clip_id": scope["clip_id"].astype(str),
                "signer_id": scope["signer_id"].astype(str),
                "label_id": scope["y_true"].astype(np.int64),
                "fold": np.full(scope["y_true"].size, fold, dtype=np.int64),
                "y_true": scope["y_true"].astype(np.int64),
                "y_pred": scope["y_pred"].astype(np.int64)}))
            probability_parts.append(scope["proba"])
        table = pd.concat(parts, ignore_index=True)
        proba = np.concatenate(probability_parts).astype(np.float32)
        order = np.argsort(table["clip_id"].to_numpy(), kind="mergesort")
        table = table.iloc[order].reset_index(drop=True)
        OOF[branch] = {
            "clip_id": table["clip_id"].to_numpy().astype(str),
            "signer_id": table["signer_id"].to_numpy().astype(str),
            "label_id": table["label_id"].to_numpy(dtype=np.int64),
            "fold": table["fold"].to_numpy(dtype=np.int64),
            "y_true": table["y_true"].to_numpy(dtype=np.int64),
            "y_pred": table["y_pred"].to_numpy(dtype=np.int64),
            "proba": proba[order],
        }

    print("OUT-OF-FOLD COVERAGE")
    Y_ALL = LABEL_OF.loc[CLIP_IDS].to_numpy(dtype=np.int64)
    SIGNER_ALL = SIGNER_OF.loc[CLIP_IDS].to_numpy().astype(str)
    OOF_DIGEST: Dict[str, str] = {}
    for branch in BRANCHES:
        block = OOF[branch]
        row_sums = block["proba"].sum(axis=1)
        print(f"  {branch:<8} rows {block['clip_id'].size:,} | unique clips "
              f"{len(set(block['clip_id'].tolist())):,} | proba "
              f"{block['proba'].shape} {block['proba'].dtype} | max |rowsum-1| "
              f"{float(np.abs(row_sums - 1.0).max()):.2e}")
        require(f"{branch} OOF holds exactly {N_CLIPS:,} rows",
                block["clip_id"].size == N_CLIPS, f"{block['clip_id'].size:,}")
        require(f"{branch} OOF covers every clip exactly once",
                sorted(block["clip_id"].tolist()) == CLIP_IDS, "no gaps, no duplicates")
        require(f"{branch} OOF labels match the frozen manifest",
                np.array_equal(block["y_true"], Y_ALL), "aligned")
        require(f"{branch} OOF signers match the frozen manifest",
                np.array_equal(block["signer_id"], SIGNER_ALL), "aligned")
        require(f"{branch} OOF probabilities are float32, finite and normalised",
                block["proba"].shape == (N_CLIPS, N_LABELS)
                and block["proba"].dtype == np.float32
                and bool(np.isfinite(block["proba"]).all())
                and float(np.abs(row_sums - 1.0).max()) < 1e-3,
                f"{float(np.abs(row_sums - 1.0).max()):.2e}")
        require(f"{branch} OOF predictions equal argmax of the probabilities",
                np.array_equal(block["y_pred"], np.argmax(block["proba"], axis=1)),
                "class order 0..199")
        require(f"{branch} every clip was predicted by the fold holding out its signer",
                all(block["signer_id"][i] in context.folds[int(block["fold"][i])]["test"]
                    for i in range(N_CLIPS)), "signer-independent by construction")
        OOF_DIGEST[branch] = array_digest(block["y_pred"], block["proba"])
        atomic_write(OOF_PATH[branch], lambda h, b=block, br=branch:
                     np.savez_compressed(
                         h, clip_id=b["clip_id"], signer_id=b["signer_id"],
                         label_id=b["label_id"], fold=b["fold"], y_true=b["y_true"],
                         y_pred=b["y_pred"], proba=b["proba"],
                         __meta__=np.array([json.dumps(
                             {**SCOPE_PROVENANCE, "branch": br,
                              "model_family": MODEL_FAMILY,
                              "kind": "signer_independent_oof", "rows": int(N_CLIPS),
                              "digest": OOF_DIGEST[br]}, sort_keys=True)])))
        with np.load(OOF_PATH[branch], allow_pickle=False) as check:
            require(f"{branch} OOF file round-trips with the same digest",
                    array_digest(check["y_pred"], check["proba"]) == OOF_DIGEST[branch],
                    OOF_DIGEST[branch])
        print(f"  wrote {OOF_PATH[branch].name}  "
              f"{OOF_PATH[branch].stat().st_size / 1e6:.1f} MB  digest "
              f"{OOF_DIGEST[branch]}")
    require("full and reduced OOF cover the same clips in the same order",
            np.array_equal(OOF["full"]["clip_id"], OOF["reduced"]["clip_id"]),
            "paired row for row")

    print("\nSIGNER-OVERLAP CONTROL (kept separate from the OOF results)")
    CONTROL_DIGEST: Dict[str, str] = {}
    for branch in BRANCHES:
        scope = SCOPE_RESULTS[(branch, "control")]
        CONTROL_DIGEST[branch] = array_digest(scope["y_pred"], scope["proba"])
        atomic_write(CONTROL_PATH[branch], lambda h, s=scope, br=branch:
                     np.savez_compressed(
                         h, clip_id=s["clip_id"], signer_id=s["signer_id"],
                         y_true=s["y_true"], y_pred=s["y_pred"], proba=s["proba"],
                         val_clip_id=s["val_clip_id"], val_y_true=s["val_y_true"],
                         val_y_pred=s["val_y_pred"], val_proba=s["val_proba"],
                         __meta__=np.array([json.dumps(
                             {**SCOPE_PROVENANCE, "branch": br,
                              "model_family": MODEL_FAMILY,
                              "kind": "INTENTIONAL_SIGNER_OVERLAP_CONTROL",
                              "label": "INTENTIONAL SIGNER-OVERLAP CONTROL",
                              "splits": CONTROL_SPLIT_SIZES,
                              "digest": CONTROL_DIGEST[br],
                              "warning": "invalid as a signer-independent evaluation; "
                                         "never pool with the 6,000 out-of-fold "
                                         "predictions"}, sort_keys=True)])))
        print(f"  {branch:<8} test {scope['y_true'].size} clips | accuracy "
              f"{scope['metrics_test']['accuracy']:.4f} | digest "
              f"{CONTROL_DIGEST[branch]}")
        require(f"{branch} control test set holds 750 clips",
                int(scope["y_true"].size) == CONTROL_SPLIT_SIZES["test"],
                int(scope["y_true"].size))
        require(f"{branch} control is labelled an intentional overlap control",
                bool(scope["is_control"]), "INTENTIONAL SIGNER-OVERLAP CONTROL")
    require("control predictions are stored apart from the OOF arrays",
            CONTROL_PATH["full"] != OOF_PATH["full"], "separate files, never pooled")

In [ ]:
# ============================================================
# CELL 10 — Metrics: per fold, validation (diagnostic), pooled, signer, class
# ============================================================
# Same schema as notebook 10 so notebook 13 can join on branch and fold.
# No inferential statistics here.

if not PIPELINE_COMPLETE:
    print("SKIPPED: the run is partial. Metrics are written only for a complete run.")
else:
    fold_rows, validation_rows = [], []
    for fold in range(N_FOLDS):
        definition = context.folds[fold]
        test_signers = list(definition["test"])
        test_clips = context.clip_ids(fold, "test")
        regimes = REGIME.loc[test_clips]
        for branch in BRANCHES:
            scope = SCOPE_RESULTS[(branch, fold)]
            metrics = scope["metrics_test"]
            row = {
                "branch": branch, "fold": fold, "model_family": MODEL_FAMILY,
                "seed": int(scope["seed"]), "parameters": PARAMETERS[branch]["total"],
                "best_epoch": int(scope["best_epoch"]),
                "epochs_run": int(scope["epochs_run"]),
                "train_clips": int(scope["n_train"]), "test_clips": int(scope["n_test"]),
                "accuracy": metrics["accuracy"], "macro_f1": metrics["macro_f1"],
                "weighted_f1": metrics["weighted_f1"],
                "balanced_accuracy": metrics["balanced_accuracy"],
                f"top{TOP_K}_accuracy": metrics[f"top{TOP_K}_accuracy"],
                "signer_macro_accuracy": metrics["signer_macro_accuracy"],
                "test_n_30fps": int((regimes == REGIME_LABELS[0]).sum()),
                "test_n_60fps": int((regimes == REGIME_LABELS[1]).sum()),
                "test_regime_mix": ("mixed" if len(set(regimes)) > 1
                                    else str(sorted(set(regimes))[0])),
                "test_signer_1": test_signers[0],
                "test_signer_1_regime": SIGNER_REGIME[test_signers[0]],
                "test_signer_2": test_signers[1],
                "test_signer_2_regime": SIGNER_REGIME[test_signers[1]],
            }
            for position, signer in enumerate(test_signers, start=1):
                row[f"test_signer_{position}_accuracy"] = metrics[
                    "per_signer_accuracy"][signer]
            fold_rows.append(row)

            validation = scope["metrics_validation"]
            validation_rows.append({
                "branch": branch, "fold": fold, "model_family": MODEL_FAMILY,
                "status": "DIAGNOSTIC ONLY",
                "validation_signers": " ".join(definition["validation"]),
                "best_epoch": int(scope["best_epoch"]), "n": validation["n"],
                "accuracy": validation["accuracy"], "macro_f1": validation["macro_f1"],
                "weighted_f1": validation["weighted_f1"],
                "balanced_accuracy": validation["balanced_accuracy"],
                f"top{TOP_K}_accuracy": validation[f"top{TOP_K}_accuracy"],
                "signer_macro_accuracy": validation["signer_macro_accuracy"],
            })

    fold_metrics = pd.DataFrame(fold_rows).sort_values(["branch", "fold"])
    validation_metrics = pd.DataFrame(validation_rows).sort_values(["branch", "fold"])

    print("PER-FOLD TEST METRICS (unseen signers)")
    print(fold_metrics[["branch", "fold", "test_signer_1", "test_signer_2",
                        "test_regime_mix", "best_epoch", "accuracy", "macro_f1",
                        f"top{TOP_K}_accuracy", "signer_macro_accuracy"]]
          .round(4).to_string(index=False))

    print("\nPAIRED FULL vs REDUCED, per fold (for notebook 13; no test is run here)")
    pivot = fold_metrics.pivot(index="fold", columns="branch", values="accuracy")
    pivot["difference_full_minus_reduced"] = pivot["full"] - pivot["reduced"]
    print(pivot.round(4).to_string())
    for branch in BRANCHES:
        values = fold_metrics.loc[fold_metrics["branch"] == branch, "accuracy"]
        print(f"  mean accuracy {branch:<8} {values.mean():.4f} +/- "
              f"{values.std(ddof=1):.4f}")
    print("  Descriptive only. The pre-specified paired sign-flip test belongs to "
          "notebook 13.")

    print("\nVALIDATION METRICS  [DIAGNOSTIC ONLY - early stopping and schedule only]")
    print(validation_metrics[["branch", "fold", "validation_signers", "best_epoch",
                             "accuracy", "macro_f1"]].round(4).to_string(index=False))

    pooled_rows, signer_rows, class_rows = [], [], []
    CONFUSION: Dict[str, np.ndarray] = {}
    for branch in BRANCHES:
        block = OOF[branch]
        metrics = scope_metrics(block["y_true"], block["y_pred"], block["proba"],
                                block["signer_id"])
        pooled_rows.append({
            "branch": branch, "model_family": MODEL_FAMILY,
            "kind": "pooled_signer_independent_oof", "n": N_CLIPS,
            "accuracy": metrics["accuracy"], "macro_f1": metrics["macro_f1"],
            "weighted_f1": metrics["weighted_f1"],
            "balanced_accuracy": metrics["balanced_accuracy"],
            f"top{TOP_K}_accuracy": metrics[f"top{TOP_K}_accuracy"],
            "signer_macro_accuracy": metrics["signer_macro_accuracy"],
            "fold_accuracy_mean": float(fold_metrics.loc[
                fold_metrics["branch"] == branch, "accuracy"].mean()),
            "fold_accuracy_std": float(fold_metrics.loc[
                fold_metrics["branch"] == branch, "accuracy"].std(ddof=1)),
            "parameters": PARAMETERS[branch]["total"],
        })
        for signer, accuracy in metrics["per_signer_accuracy"].items():
            mask = block["signer_id"] == signer
            signer_rows.append({
                "branch": branch, "model_family": MODEL_FAMILY, "signer_id": signer,
                "n": int(mask.sum()), "accuracy": accuracy,
                "regime": SIGNER_REGIME[signer],
                "fold_tested_in": int(block["fold"][mask][0]),
                f"top{TOP_K}_accuracy": top_k_accuracy(block["proba"][mask],
                                                       block["y_true"][mask])})
        recalls = recall_score(block["y_true"], block["y_pred"], average=None,
                              labels=np.arange(N_LABELS), zero_division=0)
        for label_id, recall in enumerate(recalls):
            class_rows.append({"branch": branch, "model_family": MODEL_FAMILY,
                               "label_id": label_id, "label": ID_TO_LABEL[label_id],
                               "recall": float(recall),
                               "support": int((block["y_true"] == label_id).sum())})
        CONFUSION[branch] = confusion_matrix(block["y_true"], block["y_pred"],
                                             labels=np.arange(N_LABELS)).astype(np.int32)
        np.save(CONFUSION_PATH[branch], CONFUSION[branch])
        require(f"{branch} confusion matrix is 200x200 and totals {N_CLIPS:,}",
                CONFUSION[branch].shape == (N_LABELS, N_LABELS)
                and int(CONFUSION[branch].sum()) == N_CLIPS,
                f"{CONFUSION[branch].shape}, {int(CONFUSION[branch].sum()):,}")
        require(f"{branch} pooled accuracy reproduces from the confusion matrix",
                abs(float(np.trace(CONFUSION[branch]) / N_CLIPS)
                    - metrics["accuracy"]) < 1e-12, f"{metrics['accuracy']:.6f}")

    pooled_metrics = pd.DataFrame(pooled_rows)
    signer_metrics = pd.DataFrame(signer_rows)
    class_recall = pd.DataFrame(class_rows)

    print(f"\nPOOLED {N_CLIPS:,}-CLIP OUT-OF-FOLD METRICS (the primary result)")
    print(pooled_metrics.round(4).to_string(index=False))
    print("\nPER-SIGNER ACCURACY (pooled out-of-fold)")
    print(signer_metrics.pivot(index="signer_id", columns="branch", values="accuracy")
          .reindex(sorted(SIGNER_REGIME, key=lambda v: int(v.lstrip("p"))))
          .round(4).to_string())

    control_rows = []
    for branch in BRANCHES:
        scope = SCOPE_RESULTS[(branch, "control")]
        metrics = scope["metrics_test"]
        control_rows.append({
            "branch": branch, "model_family": MODEL_FAMILY,
            "kind": "INTENTIONAL SIGNER-OVERLAP CONTROL",
            "valid_as_signer_independent": False,
            **{k: CONTROL_SPLIT_SIZES[k] for k in CONTROL_SPLIT_SIZES},
            "seed": int(scope["seed"]), "best_epoch": int(scope["best_epoch"]),
            "accuracy": metrics["accuracy"], "macro_f1": metrics["macro_f1"],
            "weighted_f1": metrics["weighted_f1"],
            "balanced_accuracy": metrics["balanced_accuracy"],
            f"top{TOP_K}_accuracy": metrics[f"top{TOP_K}_accuracy"],
            "validation_accuracy_diagnostic": scope["metrics_validation"]["accuracy"],
            "pooled_oof_accuracy_same_branch": float(
                pooled_metrics.loc[pooled_metrics["branch"] == branch,
                                   "accuracy"].iloc[0])})
    control_metrics = pd.DataFrame(control_rows)
    control_metrics["signer_overlap_gap"] = (
        control_metrics["accuracy"]
        - control_metrics["pooled_oof_accuracy_same_branch"])
    print("\nSIGNER-OVERLAP CONTROL  [invalid by design; reported as a gap]")
    print(control_metrics[["branch", "accuracy", "pooled_oof_accuracy_same_branch",
                          "signer_overlap_gap"]].round(4).to_string(index=False))
    print("  The gap is the SIGNER-OVERLAP PERFORMANCE GAP. Identity memorisation is")
    print("  one plausible contributor; session and background similarity are others.")

    efficiency = pd.DataFrame(EFFICIENCY)
    history = pd.DataFrame(HISTORY)
    preflight_rows = preflight_table.assign(scope="preflight")
    efficiency = pd.concat([efficiency, preflight_rows], ignore_index=True)

    for name, frame in (("fold_metrics", fold_metrics),
                        ("validation_metrics", validation_metrics),
                        ("pooled_metrics", pooled_metrics),
                        ("signer_metrics", signer_metrics),
                        ("class_recall", class_recall),
                        ("control_metrics", control_metrics),
                        ("efficiency", efficiency), ("history", history)):
        atomic_write(OUT[name], lambda h, f=frame: f.to_csv(h, index=False))
        print(f"  wrote {OUT[name].name}  {len(frame):,} rows")

    with np.load(OOF_PATH["full"], allow_pickle=False) as check:
        reproduced = float(accuracy_score(check["y_true"], check["y_pred"]))
    require("pooled metrics reproduce from the saved OOF file",
            abs(reproduced - float(pooled_metrics.loc[
                pooled_metrics["branch"] == "full", "accuracy"].iloc[0])) < 1e-12,
            f"{reproduced:.6f}")
    require("all eight paired fold values are saved for notebook 13",
            len(fold_metrics) == N_FOLDS * len(BRANCHES)
            and pivot["difference_full_minus_reduced"].notna().all(),
            f"{N_FOLDS} paired differences")
    require("no inferential statistic was computed here", True,
            "sign-flip test, intervals and cross-model comparison are notebook 13")

In [ ]:
# ============================================================
# CELL 11 — Freeze the manifest, then write the validation report last
# ============================================================

RUN_METADATA = {
    "generated_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "environment": ENVIRONMENT, "determinism": DETERMINISM_NOTE,
    "resolved_paths_this_session": {"notebook8": str(NB8_DIR), "notebook9": str(NB9_DIR)},
    "resume_sources": [str(d) for d in RESUME_DIRS],
    "runtime_seconds": round(time.time() - STARTED, 1),
    "note": "recorded for traceability and deliberately excluded from "
            "nn_model_contract_id, which must not change between reruns",
}

if not PIPELINE_COMPLETE:
    remaining = [scope_label(b, s) for b, s in ALL_SCOPES if (b, s) not in SCOPE_RESULTS]
    report = {
        "notebook": "11-bilstm-8-fold", "model_family": MODEL_FAMILY,
        "verdict": "PARTIAL_RESUME_REQUIRED",
        "nn_model_contract_id": NN_MODEL_CONTRACT_ID,
        "upstream": dict(SCOPE_PROVENANCE),
        "scopes_total": len(ALL_SCOPES), "scopes_complete": len(SCOPE_RESULTS),
        "scopes_remaining": remaining,
        "checks_total": len(CHECKS),
        "checks_passed": len([c for c in CHECKS if c["status"] == "pass"]),
        "checks_failed": len(failures()),
        "checks_skipped": len([c for c in CHECKS if c["status"] == "skip"]),
        "checks": CHECKS, "efficiency": EFFICIENCY,
        "runtime_seconds": round(time.time() - STARTED, 1),
        "run_metadata": RUN_METADATA,
    }
    atomic_write(OUT["report_json"], lambda h: h.write(
        json.dumps(report, indent=2, default=str).encode()))
    atomic_write(OUT["report_csv"], lambda h: pd.DataFrame(CHECKS)[
        ["check", "status", "passed", "detail"]].to_csv(h, index=False))
    print("=" * 78)
    print("NOTEBOOK 11 PARTIAL - RESUME REQUIRED")
    print("=" * 78)
    print(f"  nn model contract id : {NN_MODEL_CONTRACT_ID}")
    print(f"  scopes complete      : {len(SCOPE_RESULTS)} of {len(ALL_SCOPES)}")
    print(f"  remaining            : {', '.join(remaining)}")
    print(f"  checks               : {report['checks_passed']} passed, "
          f"{report['checks_failed']} failed of {len(CHECKS)}")
    print(f"  runtime              : {(time.time() - STARTED) / 60:.1f} min")
    print("\n  Commit this version, attach its output as an extra input, and rerun.")
    print("  Completed scopes are re-verified and skipped. No result is lost.")
    print("=" * 78)
else:
    pooled_lookup = {row["branch"]: row for row in pooled_metrics.to_dict("records")}
    control_lookup = {row["branch"]: row for row in control_metrics.to_dict("records")}

    MANIFEST = {
        "nn_model_contract_id": NN_MODEL_CONTRACT_ID,
        "nn_model_contract": NN_MODEL_CONTRACT,
        "model_family": MODEL_FAMILY,
        "upstream": dict(SCOPE_PROVENANCE),
        "scopes_completed": len(SCOPE_RESULTS),
        "parameters": PARAMETERS,
        "total_parameter_difference": int(total_difference),
        "oof": {b: {"file": OOF_PATH[b].name, "rows": N_CLIPS,
                    "digest": OOF_DIGEST[b],
                    "accuracy": pooled_lookup[b]["accuracy"],
                    "macro_f1": pooled_lookup[b]["macro_f1"],
                    "signer_macro_accuracy": pooled_lookup[b]["signer_macro_accuracy"]}
                for b in BRANCHES},
        "control": {b: {"file": CONTROL_PATH[b].name,
                        "label": "INTENTIONAL SIGNER-OVERLAP CONTROL",
                        "digest": CONTROL_DIGEST[b],
                        "accuracy": control_lookup[b]["accuracy"],
                        "signer_overlap_gap": control_lookup[b]["signer_overlap_gap"],
                        "valid_as_signer_independent": False} for b in BRANCHES},
        "paired_fold_accuracy": {
            "full": [float(v) for v in fold_metrics.loc[
                fold_metrics["branch"] == "full"].sort_values("fold")["accuracy"]],
            "reduced": [float(v) for v in fold_metrics.loc[
                fold_metrics["branch"] == "reduced"].sort_values("fold")["accuracy"]],
            "note": "saved for the pre-specified paired test in notebook 13; no test is "
                    "run in notebook 11"},
        "best_epochs": {b: {str(f): int(SCOPE_RESULTS[(b, f)]["best_epoch"])
                            for f in list(range(N_FOLDS)) + ["control"]}
                        for b in BRANCHES},
        "artefacts": {name: path.name for name, path in OUT.items()},
        "confusion": {b: CONFUSION_PATH[b].name for b in BRANCHES},
        "run_metadata": RUN_METADATA,
    }
    atomic_write(OUT["manifest"], lambda h: h.write(
        json.dumps(MANIFEST, indent=2, sort_keys=True, default=str).encode()))
    reloaded_manifest = json.loads(OUT["manifest"].read_text(encoding="utf-8"))
    require("the neural model contract id reproduces from the manifest on disk",
            sha16(json.dumps(reloaded_manifest["nn_model_contract"], sort_keys=True,
                             separators=(",", ":"), default=str))
            == NN_MODEL_CONTRACT_ID == reloaded_manifest["nn_model_contract_id"],
            NN_MODEL_CONTRACT_ID)
    require("run metadata is stored outside the hashed contract",
            "generated_at_utc" not in json.dumps(
                reloaded_manifest["nn_model_contract"]), "separated")
    require("the manifest records both OOF digests",
            all(reloaded_manifest["oof"][b]["digest"] == OOF_DIGEST[b]
                for b in BRANCHES), "recorded")
    require("the manifest records the parameter difference, not equality",
            reloaded_manifest["total_parameter_difference"] == 288768, "288,768")

    FINAL_FAILURES = failures()
    SKIPPED = [entry for entry in CHECKS if entry["status"] == "skip"]
    VERDICT = "FAIL" if FINAL_FAILURES else "PASS"
    report = {
        "notebook": "11-bilstm-8-fold", "model_family": MODEL_FAMILY,
        "nn_policy_version": NN_POLICY_VERSION, "verdict": VERDICT,
        "nn_model_contract_id": NN_MODEL_CONTRACT_ID,
        "upstream": dict(SCOPE_PROVENANCE),
        "checks_total": len(CHECKS),
        "checks_passed": len([c for c in CHECKS if c["status"] == "pass"]),
        "checks_failed": len(FINAL_FAILURES), "checks_skipped": len(SKIPPED),
        "checks": CHECKS,
        "oof_digest": OOF_DIGEST, "control_digest": CONTROL_DIGEST,
        "pooled_metrics": pooled_metrics.to_dict("records"),
        "control_metrics": control_metrics.to_dict("records"),
        "fold_metrics": fold_metrics.to_dict("records"),
        "efficiency": efficiency.to_dict("records"),
        "runtime_seconds": round(time.time() - STARTED, 1),
        "run_metadata": RUN_METADATA,
    }
    atomic_write(OUT["report_json"], lambda h: h.write(
        json.dumps(report, indent=2, default=str).encode()))
    atomic_write(OUT["report_csv"], lambda h: pd.DataFrame(CHECKS)[
        ["check", "status", "passed", "detail"]].to_csv(h, index=False))
    print(f"wrote {OUT['report_json'].name} and {OUT['report_csv'].name} "
          f"({len(CHECKS)} checks, verdict {VERDICT})")

    if FINAL_FAILURES:
        for entry in FINAL_FAILURES:
            print(f"  FAIL  {entry['check']}: {entry['detail']}")
        raise RuntimeError(
            f"{len(FINAL_FAILURES)} checks failed. The report was written for "
            "diagnosis, but these BiLSTM results are NOT usable and notebook 13 must "
            "not read them.")

    written = list(dict.fromkeys(
        [p for p in list(OUT.values()) + list(OOF_PATH.values())
         + list(CONTROL_PATH.values()) + list(CONFUSION_PATH.values()) if p.exists()]))
    scope_bytes = sum(p.stat().st_size for p in SCOPE_DIR.glob("nn_scope_*.npz"))
    total_bytes = sum(p.stat().st_size for p in written) + scope_bytes

    print("=" * 78)
    print("NOTEBOOK 11 COMPLETE - BiLSTM EVALUATION FROZEN")
    print("=" * 78)
    print(f"  experiment id        : {REQUIRED_EXPERIMENT_ID}")
    print(f"  loader contract id   : {REQUIRED_LOADER_CONTRACT_ID}")
    print(f"  standardiser set id  : {REQUIRED_STANDARDISER_SET_ID}")
    print(f"  time metadata id     : {REQUIRED_TIME_METADATA_ID}")
    print(f"  nn model contract id : {NN_MODEL_CONTRACT_ID}")
    print()
    print(f"  full    OOF digest    : {OOF_DIGEST['full']}")
    print(f"  reduced OOF digest    : {OOF_DIGEST['reduced']}")
    print(f"  full    control digest: {CONTROL_DIGEST['full']}")
    print(f"  reduced control digest: {CONTROL_DIGEST['reduced']}")
    print()
    print(f"  POOLED {N_CLIPS:,}-CLIP OUT-OF-FOLD (primary result)")
    for branch in BRANCHES:
        row = pooled_lookup[branch]
        print(f"    {branch:<8} accuracy {row['accuracy']:.4f} | macro F1 "
              f"{row['macro_f1']:.4f} | weighted F1 {row['weighted_f1']:.4f} | "
              f"balanced {row['balanced_accuracy']:.4f}")
        print(f"    {'':<8} top{TOP_K} {row[f'top{TOP_K}_accuracy']:.4f} | "
              f"signer-macro {row['signer_macro_accuracy']:.4f} | per-fold "
              f"{row['fold_accuracy_mean']:.4f} +/- {row['fold_accuracy_std']:.4f} | "
              f"{row['parameters']:,} parameters")
    print()
    print("  SIGNER-OVERLAP CONTROL (invalid by design, reported as a gap)")
    for branch in BRANCHES:
        row = control_lookup[branch]
        print(f"    {branch:<8} accuracy {row['accuracy']:.4f} | gap vs pooled OOF "
              f"{row['signer_overlap_gap']:+.4f}")
    print()
    print(f"  checks               : {report['checks_passed']} passed, "
          f"{report['checks_failed']} failed, {report['checks_skipped']} skipped "
          f"of {len(CHECKS)}")
    print(f"  runtime              : {(time.time() - STARTED) / 60:.1f} min")
    print(f"  outputs              : {total_bytes / 1e6:.0f} MB "
          "(no trained network is persisted)")

    print("\n  Frozen here: the BiLSTM model contract, the 6,000-clip out-of-fold")
    print("  predictions and probabilities for both branches, the two control runs,")
    print("  per-fold, pooled, per-signer and per-class metrics, confusion matrices,")
    print("  training histories and efficiency measurements.")
    print("\n  NOT done here: no paired sign-flip test, no confidence interval, no")
    print("  equivalence claim, no cross-model comparison. Notebook 13 owns all of that.")
    print("\n  Carried forward as interpretation constraints:")
    print("    - recurrent core matched, total parameter count NOT identical: the full")
    print(f"      branch carries {total_difference:,} extra input-projection weights.")
    print("    - NATIVE FRAMES were used. The ~30/~60 fps regimes track the signer, so")
    print("      sequence length is a signer-correlated nuisance. The common-time")
    print("      sensitivity analysis is pre-declared as secondary, its exact method")
    print("      is not yet frozen, and it was not run here.")
    print("    - the Random Forest branch used physical-time motion features while this")
    print("      branch used native frames; notebook 13 must state that when comparing.")

    print("\n" + "=" * 78)
    print(f"  Quote nn model contract id {NN_MODEL_CONTRACT_ID} in notebook 13.")
    print("  STOP. Do not build notebook 12 until this real Kaggle log has been "
          "reviewed.")
    print("=" * 78)